In [ ]:
!pip install -U numpy scikit-learn joblib tqdm rasterio geopandas shapely pyproj fiona shap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.7/676.7 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 103.0 MB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


# RFECV and feature Importance for GSE (Dataset-II)

In [2]:
# Optional Colab installation (run once if required):
# !pip install -q geopandas pyogrio shap joblib

"""
Spatial RFECV + Random Forest grid search for SOC prediction.

Main design
-----------
1. The 90% development dataset is used for:
   - correlation filtering;
   - RFECV feature selection;
   - spatial cross-validation grid-search evaluation;
   - fitting the final best model.
2. Every parameter combination uses the same Random Forest settings:
   - inside RFECV;
   - during spatial CV evaluation;
   - during final model fitting.
3. The best combination is selected using development spatial CV only:
   - highest out-of-fold R2;
   - lowest out-of-fold RMSE as the tie-breaker;
   - lowest MAE and fewer selected features as additional tie-breakers.
4. The independent 10% validation dataset is evaluated only once,
   after the best combination has been selected.
5. Detailed files are saved only for the best parameter combination.
   The only all-combination output is grid_search_all_results.csv.
6. After the best parameter combination is identified, two models are
   compared using exactly the same best Random Forest and spatial-CV
   settings:
   - all original predictor variables, without correlation filtering or RFECV;
   - the correlation-filtered and RFECV-selected variables.
   The all-variable model is not tuned separately; this isolates the effect
   of feature selection under the selected best model configuration.
"""

import gc
import json
import logging
import math
import time
import traceback
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFECV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, ParameterGrid


# ============================================================
# USER SETTINGS
# ============================================================

# 90% development dataset used for feature selection and grid search.
DEVELOPMENT_GPKG = (
    "/content/drive/MyDrive/1.SOC_Estimation/"
    "1.Final_SOC_estimation/1_Feature_Selection/Split_features/"
    "LUCAS_samples_GSE_train_test.gpkg"
)

# Untouched 10% independent validation dataset.
INDEPENDENT_VALIDATION_GPKG = (
    "/content/drive/MyDrive/1.SOC_Estimation/"
    "1.Final_SOC_estimation/1_Feature_Selection/Split_features/"
    "LUCAS_samples_GSE_validation.gpkg"
)

OUT_DIR = (
    "/content/drive/MyDrive/1.SOC_Estimation/"
    "1.Final_SOC_estimation/1_Feature_Selection/"
    "RFECV_SOC_Grid_Search"
)

TARGET_COL = "OC"
TARGET_CRS = "EPSG:3763"

# ------------------------------------------------------------
# Predictor-selection rule
# ------------------------------------------------------------
# "prefix": use all columns beginning with PREDICTOR_PREFIX.
# This preserves the rule from your uploaded GSE workflow.
#
# "all_numeric_except": use every numeric attribute except the
# target, geometry, and EXCLUDED_PREDICTOR_COLUMNS. This can be
# used for the Sentinel workflow after unwanted ID fields have
# already been removed during preprocessing.
PREDICTOR_MODE = "prefix"
PREDICTOR_PREFIX = "A"

EXCLUDED_PREDICTOR_COLUMNS = [
    "LC0_Desc",
    "LC1_Desc",
    "LU1_Desc",
]

# Missing-value handling.
DROP_ROWS_WITH_NAN = True
FAIL_ON_NONNUMERIC_PREDICTORS = True

# Fixed grid origin for spatial blocks.
GRID_OFFSET_X_M = 0
GRID_OFFSET_Y_M = 0

# RFECV settings that are not included in the grid.
RFECV_SCORING = "neg_root_mean_squared_error"
RF_IMPORTANCE_GETTER = "auto"

# SHAP settings for the final best model only.
MAX_SHAP_SAMPLES = 1000
TOP_N_SHAP_FEATURES = 20

# Output image resolution.
OUTPUT_DPI = 300

# Safety control. RFECV is computationally expensive for every grid row.
MAX_ALLOWED_GRID_COMBINATIONS = 200
ALLOW_LARGE_GRID = False

# Save grid_search_all_results.csv after every attempted combination so
# completed work remains available if Colab disconnects or the runtime stops.
CHECKPOINT_AFTER_EACH_COMBINATION = True

# Automatically inspect grid_search_all_results.csv and continue with the
# first parameter combination that has not completed successfully.
RESUME_GRID_SEARCH = True

# True: combinations recorded as "failed" are attempted again on the next run.
# False: both successful and failed combinations are skipped.
RETRY_FAILED_COMBINATIONS = True

# Write each checkpoint through a temporary file and then replace the main CSV.
# This reduces the risk of leaving a partially written CSV after interruption.
ATOMIC_CHECKPOINT_WRITES = True

# Keep only checkpoint rows belonging to the current PARAMETER_GRID.
# Rows from older/different grids are ignored but the original CSV is not
# deleted; the next checkpoint rewrites it with the current-grid rows.
IGNORE_RESULTS_OUTSIDE_CURRENT_GRID = True


# ============================================================
# PARAMETER GRID
# ============================================================
# No baseline scenario is used. Every Cartesian combination below
# is treated as a candidate configuration.
#
# IMPORTANT RUNTIME NOTE:
# The default grid contains 64 combinations. Every combination runs
# an RFECV procedure plus a full spatial cross-validation evaluation.
# Increase the grid only when sufficient runtime is available.

PARAMETER_GRID = {
    # Random Forest randomness
    "random_seed": [42, 123],

    # Spatial cross-validation design
    "block_size_m": [15_000, 20_000],
    "n_spatial_folds": [5, 10],

    # Pre-RFECV correlation filter
    "corr_threshold": [0.95, 0.99],

    # Random Forest parameters. The exact same values are used inside
    # RFECV, during CV evaluation, and in the final fitted model.
    "n_estimators": [1000, 2000],
    "max_depth": [None],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt"],
    "bootstrap": [True],

    # RFECV search settings
    "rfecv_step": [1],
    "min_features_to_select": [1],
}


# ============================================================
# OUTPUT FILES
# ============================================================

OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = OUT_DIR / "RFECV_SOC_grid_search.log"

# Only file containing results for every candidate combination.
OUT_ALL_RESULTS_CSV = OUT_DIR / "grid_search_all_results.csv"

# Best-parameter documentation.
OUT_BEST_PARAMETERS_CSV = OUT_DIR / "best_parameters.csv"
OUT_ATTRIBUTE_INVENTORY_CSV = OUT_DIR / "best_input_attribute_inventory.csv"

# Best feature-selection outputs.
OUT_CORRELATION_CSV = OUT_DIR / "best_correlation_filter_results.csv"
OUT_SELECTED_CSV = OUT_DIR / "best_RFECV_selected_features.csv"
OUT_RFECV_CURVE_CSV = OUT_DIR / "best_RFECV_curve.csv"
OUT_RFECV_CURVE_PNG = OUT_DIR / "best_RFECV_curve.png"
OUT_SPATIAL_FOLD_METRICS_CSV = OUT_DIR / "best_spatial_cv_fold_metrics.csv"

# Best feature-importance outputs.
OUT_SHAP_IMPORTANCE_CSV = OUT_DIR / "best_SHAP_feature_importance.csv"
OUT_SHAP_TOP20_CSV = OUT_DIR / "best_SHAP_top20_feature_importance.csv"
OUT_RF_IMPORTANCE_CSV = OUT_DIR / "best_RF_feature_importance.csv"
OUT_SHAP_PNG = OUT_DIR / "best_SHAP_beeswarm_top20.png"

# Best model evaluation and spatial outputs.
OUT_METRICS_SUMMARY_CSV = OUT_DIR / "best_model_metrics_summary.csv"
OUT_VALIDATION_PREDICTIONS_GPKG = (
    OUT_DIR / "best_independent_validation_predictions.gpkg"
)
OUT_BLOCKS_GPKG = OUT_DIR / "best_development_points_spatial_folds.gpkg"
OUT_FINAL_MODEL = OUT_DIR / "best_RF_selected_features.joblib"

# All-features versus selected-features comparison using the same best
# parameter combination. No correlation filtering or RFECV is applied to
# the all-features model.
OUT_ALL_FEATURES_FOLD_METRICS_CSV = (
    OUT_DIR / "best_all_features_spatial_cv_fold_metrics.csv"
)
OUT_FOLD_COMPARISON_CSV = (
    OUT_DIR / "best_all_vs_selected_fold_comparison.csv"
)
OUT_FEATURE_SELECTION_COMPARISON_CSV = (
    OUT_DIR / "best_all_vs_selected_metrics_comparison.csv"
)
OUT_FEATURE_SELECTION_IMPACT_CSV = (
    OUT_DIR / "best_feature_selection_impact_summary.csv"
)
OUT_FEATURE_SELECTION_COMPARISON_PNG = (
    OUT_DIR / "best_all_vs_selected_metrics_comparison.png"
)
OUT_ALL_FEATURES_MODEL = (
    OUT_DIR / "best_RF_all_features_same_parameters.joblib"
)


# ============================================================
# LOGGING
# ============================================================

def setup_logger(log_file):
    """Configure console logging and append to the log when resuming."""
    logger = logging.getLogger("soc_rfecv_grid_search")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

    log_path = Path(log_file)
    append_existing_log = bool(
        RESUME_GRID_SEARCH and log_path.exists()
    )

    file_handler = logging.FileHandler(
        log_path,
        mode="a" if append_existing_log else "w",
        encoding="utf-8",
    )
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    if append_existing_log:
        logger.info("")
        logger.info("=" * 90)
        logger.info("NEW RESUME SESSION")
        logger.info("=" * 90)

    return logger


logger = setup_logger(LOG_FILE)


# ============================================================
# GENERAL HELPERS
# ============================================================

def print_dataframe(title, dataframe, max_rows=None):
    """Print a DataFrame as a documentation table."""
    print("\n" + "=" * 120)
    print(title)
    print("=" * 120)

    if dataframe.empty:
        print("No records.")
        return

    display_df = dataframe if max_rows is None else dataframe.head(max_rows)

    with pd.option_context(
        "display.max_columns", None,
        "display.width", 300,
        "display.max_colwidth", 80,
    ):
        print(display_df.to_string(index=False))

    if max_rows is not None and len(dataframe) > max_rows:
        print(f"\nShowing {max_rows:,} of {len(dataframe):,} records.")


def printable_value(value):
    """Convert values such as None into clear table text."""
    return "None" if value is None else value


def remove_existing_file(path):
    """Remove an existing file before rewriting it."""
    path = Path(path)
    if path.exists():
        path.unlink()


def build_case_insensitive_lookup(columns):
    """Map case-insensitive names to the actual field names."""
    lookup = {}

    for column in columns:
        key = str(column).strip().casefold()

        if key in lookup:
            raise ValueError(
                "Duplicate field names were found when compared without "
                f"case sensitivity: '{lookup[key]}' and '{column}'."
            )

        lookup[key] = column

    return lookup


def combination_signature(config):
    """
    Create a stable signature from model/search parameters only.

    grid_id is intentionally excluded, so the same parameter combination
    can be recognized even if the order of PARAMETER_GRID changes.
    """
    ordered = {
        key: printable_value(config[key])
        for key in parameter_columns()
    }
    return json.dumps(ordered, sort_keys=True, separators=(",", ":"))


def parameter_columns():
    """Return the ordered parameter columns used in output tables."""
    return [
        "random_seed",
        "block_size_m",
        "n_spatial_folds",
        "corr_threshold",
        "n_estimators",
        "max_depth",
        "min_samples_leaf",
        "max_features",
        "bootstrap",
        "rfecv_step",
        "min_features_to_select",
    ]


INTEGER_PARAMETER_COLUMNS = {
    "random_seed",
    "block_size_m",
    "n_spatial_folds",
    "n_estimators",
    "min_samples_leaf",
    "rfecv_step",
    "min_features_to_select",
}

FLOAT_PARAMETER_COLUMNS = {
    "corr_threshold",
}

BOOLEAN_PARAMETER_COLUMNS = {
    "bootstrap",
}


def parse_boolean(value):
    """Convert common CSV boolean representations to bool."""
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    text = str(value).strip().casefold()

    if text in {"true", "1", "yes", "y"}:
        return True

    if text in {"false", "0", "no", "n"}:
        return False

    raise ValueError(f"Cannot interpret '{value}' as a Boolean value.")


def normalize_checkpoint_parameter(column, value):
    """
    Recover the expected Python type for a parameter read from CSV.
    """
    if column in INTEGER_PARAMETER_COLUMNS:
        return int(float(value))

    if column in FLOAT_PARAMETER_COLUMNS:
        return float(value)

    if column in BOOLEAN_PARAMETER_COLUMNS:
        return parse_boolean(value)

    if column == "max_depth":
        if pd.isna(value):
            return None

        text = str(value).strip()

        if text.casefold() in {"none", "nan", ""}:
            return None

        return int(float(value))

    if column == "max_features":
        if pd.isna(value):
            return None

        text = str(value).strip()

        if text.casefold() in {"none", "nan", ""}:
            return None

        # sklearn accepts named strings such as "sqrt", or numeric values.
        try:
            return float(text)
        except ValueError:
            return text

    return value


def signature_from_checkpoint_row(row):
    """
    Rebuild a current-format signature from checkpoint parameter columns.

    This also makes checkpoints created by an older script version usable
    when their saved parameter_signature included grid_id.
    """
    missing_columns = [
        column
        for column in parameter_columns()
        if column not in row.index
    ]

    if missing_columns:
        raise KeyError(
            "Checkpoint is missing parameter columns: "
            f"{missing_columns}"
        )

    config = {
        column: normalize_checkpoint_parameter(
            column,
            row[column],
        )
        for column in parameter_columns()
    }

    return combination_signature(config)


def checkpoint_is_successful(row):
    """Return True only for a complete, valid successful result row."""
    status = str(row.get("status", "")).strip().casefold()

    if status != "success":
        return False

    cv_r2 = pd.to_numeric(
        pd.Series([row.get("CV_R2", np.nan)]),
        errors="coerce",
    ).iloc[0]

    cv_rmse = pd.to_numeric(
        pd.Series([row.get("CV_RMSE", np.nan)]),
        errors="coerce",
    ).iloc[0]

    selected_count = pd.to_numeric(
        pd.Series([row.get("n_selected_features", np.nan)]),
        errors="coerce",
    ).iloc[0]

    return bool(
        pd.notna(cv_r2)
        and pd.notna(cv_rmse)
        and pd.notna(selected_count)
        and selected_count >= 1
    )


def write_checkpoint_atomic(records, output_csv):
    """
    Save checkpoint rows safely, with one latest row per signature.
    """
    checkpoint_df = pd.DataFrame(records)

    if checkpoint_df.empty:
        return

    if "parameter_signature" not in checkpoint_df.columns:
        raise KeyError(
            "Cannot save checkpoint without parameter_signature."
        )

    checkpoint_df = (
        checkpoint_df
        .drop_duplicates(
            subset=["parameter_signature"],
            keep="last",
        )
        .reset_index(drop=True)
    )

    output_csv = Path(output_csv)

    if ATOMIC_CHECKPOINT_WRITES:
        temporary_csv = output_csv.with_name(
            output_csv.stem + "_writing.tmp.csv"
        )

        checkpoint_df.to_csv(
            temporary_csv,
            index=False,
        )

        temporary_csv.replace(output_csv)

    else:
        checkpoint_df.to_csv(
            output_csv,
            index=False,
        )


def load_resume_checkpoint(output_csv, current_combinations):
    """
    Load and validate grid_search_all_results.csv.

    Successful combinations are skipped. Failed or incomplete combinations
    are rerun when RETRY_FAILED_COMBINATIONS=True.
    """
    output_csv = Path(output_csv)

    if not RESUME_GRID_SEARCH or not output_csv.exists():
        return [], set(), set()

    try:
        checkpoint_df = pd.read_csv(output_csv)
    except Exception as error:
        raise RuntimeError(
            "The existing grid_search_all_results.csv could not be read. "
            "Rename or repair it before resuming. Original error: "
            f"{error}"
        ) from error

    if checkpoint_df.empty:
        logger.warning(
            "Existing checkpoint CSV is empty. Starting the full grid."
        )
        return [], set(), set()

    required_columns = {
        "status",
        *parameter_columns(),
    }

    missing_columns = required_columns.difference(
        checkpoint_df.columns
    )

    if missing_columns:
        raise ValueError(
            "Existing grid_search_all_results.csv is missing required "
            f"columns: {sorted(missing_columns)}"
        )

    rebuilt_signatures = []
    invalid_row_indices = []

    for row_index, row in checkpoint_df.iterrows():
        try:
            rebuilt_signatures.append(
                signature_from_checkpoint_row(row)
            )
        except Exception as error:
            logger.warning(
                "Ignoring checkpoint row %s because its parameter values "
                "could not be reconstructed: %s",
                row_index,
                error,
            )
            rebuilt_signatures.append(None)
            invalid_row_indices.append(row_index)

    checkpoint_df["parameter_signature"] = rebuilt_signatures

    checkpoint_df = checkpoint_df.loc[
        checkpoint_df["parameter_signature"].notna()
    ].copy()

    current_signatures = {
        config["parameter_signature"]
        for config in current_combinations
    }

    outside_grid_mask = (
        ~checkpoint_df["parameter_signature"].isin(
            current_signatures
        )
    )

    outside_grid_count = int(outside_grid_mask.sum())

    if outside_grid_count > 0:
        message = (
            f"{outside_grid_count} checkpoint rows do not belong to the "
            "current PARAMETER_GRID."
        )

        if IGNORE_RESULTS_OUTSIDE_CURRENT_GRID:
            logger.warning(
                "%s They will be ignored during this run.",
                message,
            )
            checkpoint_df = checkpoint_df.loc[
                ~outside_grid_mask
            ].copy()
        else:
            raise ValueError(
                message
                + " Set IGNORE_RESULTS_OUTSIDE_CURRENT_GRID=True "
                "or use a new output folder."
            )

    # Prefer a valid success row over a failed duplicate. Otherwise retain
    # the latest occurrence for that signature.
    checkpoint_df["_valid_success"] = checkpoint_df.apply(
        checkpoint_is_successful,
        axis=1,
    )

    checkpoint_df["_source_order"] = np.arange(
        len(checkpoint_df)
    )

    checkpoint_df = (
        checkpoint_df
        .sort_values(
            ["parameter_signature", "_valid_success", "_source_order"],
            ascending=[True, True, True],
        )
        .drop_duplicates(
            subset=["parameter_signature"],
            keep="last",
        )
        .drop(
            columns=["_valid_success", "_source_order"],
        )
        .reset_index(drop=True)
    )

    successful_signatures = set()
    failed_or_incomplete_signatures = set()

    for _, row in checkpoint_df.iterrows():
        signature = row["parameter_signature"]

        if checkpoint_is_successful(row):
            successful_signatures.add(signature)
        else:
            failed_or_incomplete_signatures.add(signature)

    if RETRY_FAILED_COMBINATIONS:
        # Remove old failed rows before retrying so each signature has only
        # the latest attempt in the rewritten checkpoint.
        retained_df = checkpoint_df.loc[
            checkpoint_df["parameter_signature"].isin(
                successful_signatures
            )
        ].copy()
    else:
        retained_df = checkpoint_df.copy()

    logger.info(
        "Checkpoint found: %d successful, %d failed/incomplete, "
        "%d remaining current-grid combinations.",
        len(successful_signatures),
        len(failed_or_incomplete_signatures),
        len(current_signatures - successful_signatures),
    )

    if RETRY_FAILED_COMBINATIONS and failed_or_incomplete_signatures:
        logger.info(
            "Failed/incomplete combinations will be retried: %s",
            sorted(failed_or_incomplete_signatures),
        )

    records = retained_df.to_dict("records")

    skipped_signatures = set(successful_signatures)

    if not RETRY_FAILED_COMBINATIONS:
        skipped_signatures.update(
            failed_or_incomplete_signatures
        )

    return (
        records,
        skipped_signatures,
        failed_or_incomplete_signatures,
    )


# ============================================================
# DATA-LOADING HELPERS
# ============================================================

def standardize_target_name(gdf, dataset_name):
    """Find the target field case-insensitively and standardize its name."""
    lookup = build_case_insensitive_lookup(gdf.columns)
    target_key = TARGET_COL.casefold()

    if target_key not in lookup:
        raise ValueError(
            f"{dataset_name} does not contain target field '{TARGET_COL}'."
        )

    actual_target_name = lookup[target_key]

    if actual_target_name != TARGET_COL:
        gdf = gdf.rename(columns={actual_target_name: TARGET_COL})
        logger.warning(
            "%s: renamed target field '%s' to '%s'.",
            dataset_name,
            actual_target_name,
            TARGET_COL,
        )

    return gdf


def get_predictor_columns(gdf):
    """Select model predictors according to PREDICTOR_MODE."""
    geometry_column = gdf.geometry.name

    if PREDICTOR_MODE == "prefix":
        predictors = [
            column
            for column in gdf.columns
            if str(column).upper().startswith(PREDICTOR_PREFIX.upper())
            and str(column).casefold() != str(geometry_column).casefold()
            and str(column).casefold() != TARGET_COL.casefold()
        ]

    elif PREDICTOR_MODE == "all_numeric_except":
        excluded_keys = {
            TARGET_COL.casefold(),
            str(geometry_column).casefold(),
            *[
                str(column).strip().casefold()
                for column in EXCLUDED_PREDICTOR_COLUMNS
            ],
        }

        numeric_columns = (
            gdf.select_dtypes(include=[np.number])
            .columns
            .tolist()
        )

        predictors = [
            column
            for column in numeric_columns
            if str(column).strip().casefold() not in excluded_keys
        ]

    else:
        raise ValueError(
            "PREDICTOR_MODE must be either 'prefix' or "
            "'all_numeric_except'."
        )

    if not predictors:
        raise ValueError(
            "No predictor columns were found using the configured "
            f"PREDICTOR_MODE='{PREDICTOR_MODE}'."
        )

    return predictors


def align_required_predictors(gdf, required_predictors, dataset_name):
    """Align validation field names with development predictor names."""
    lookup = build_case_insensitive_lookup(gdf.columns)
    rename_map = {}
    missing = []

    for predictor in required_predictors:
        key = str(predictor).strip().casefold()

        if key not in lookup:
            missing.append(predictor)
            continue

        actual_name = lookup[key]

        if actual_name != predictor:
            rename_map[actual_name] = predictor

    if missing:
        raise ValueError(
            f"{dataset_name} is missing required predictors:\n"
            + "\n".join(f"  - {column}" for column in missing)
        )

    if rename_map:
        gdf = gdf.rename(columns=rename_map)
        logger.warning(
            "%s: aligned field names to development names: %s",
            dataset_name,
            rename_map,
        )

    return gdf


def convert_required_columns_to_numeric(
    gdf,
    required_columns,
    dataset_name,
):
    """Convert target and predictors to finite numeric values."""
    result = gdf.copy()
    conversion_failures = {}

    for column in required_columns:
        original = result[column]
        converted = pd.to_numeric(original, errors="coerce")

        failed_mask = original.notna() & converted.isna()
        failed_count = int(failed_mask.sum())

        if failed_count > 0:
            examples = (
                original.loc[failed_mask]
                .astype(str)
                .drop_duplicates()
                .head(5)
                .tolist()
            )
            conversion_failures[column] = {
                "count": failed_count,
                "examples": examples,
            }

        result[column] = converted.astype(np.float64)

    if conversion_failures:
        lines = [
            f"  - {column}: {details['count']:,} values; "
            f"examples={details['examples']}"
            for column, details in conversion_failures.items()
        ]

        message = (
            f"{dataset_name} contains values that cannot be converted "
            "to numeric form:\n" + "\n".join(lines)
        )

        if FAIL_ON_NONNUMERIC_PREDICTORS:
            raise ValueError(message)

        logger.warning(message)

    result[required_columns] = result[required_columns].replace(
        [np.inf, -np.inf],
        np.nan,
    )

    rows_before = len(result)

    if DROP_ROWS_WITH_NAN:
        result = result.dropna(subset=required_columns).copy()
        rows_removed = rows_before - len(result)

        if rows_removed > 0:
            logger.warning(
                "%s: removed %d rows with missing target or predictors.",
                dataset_name,
                rows_removed,
            )

    elif result[required_columns].isna().any().any():
        missing_counts = result[required_columns].isna().sum()
        missing_counts = missing_counts[missing_counts > 0]

        raise ValueError(
            f"{dataset_name} contains missing target or predictor values:\n"
            + missing_counts.to_string()
        )

    if result.empty:
        raise ValueError(
            f"No complete observations remain in {dataset_name}."
        )

    return result


def load_points(gpkg_path, dataset_name, required_predictors=None):
    """Read, validate, reproject, and prepare a point GeoPackage."""
    gpkg_path = Path(gpkg_path)

    if not gpkg_path.exists():
        raise FileNotFoundError(
            f"{dataset_name} GeoPackage was not found:\n{gpkg_path}"
        )

    gdf = gpd.read_file(gpkg_path)

    if gdf.empty:
        raise ValueError(f"{dataset_name} contains no features.")

    gdf = standardize_target_name(gdf, dataset_name)

    if gdf.crs is None:
        raise ValueError(
            f"{dataset_name} has no CRS information."
        )

    valid_geometry_mask = (
        gdf.geometry.notna()
        & (~gdf.geometry.is_empty)
    )

    removed_geometry_rows = int((~valid_geometry_mask).sum())

    if removed_geometry_rows > 0:
        logger.warning(
            "%s: removed %d rows with missing or empty geometry.",
            dataset_name,
            removed_geometry_rows,
        )

    gdf = gdf.loc[valid_geometry_mask].copy()

    invalid_geometry_mask = ~gdf.geometry.geom_type.isin(["Point"])

    if invalid_geometry_mask.any():
        invalid_types = sorted(
            gdf.loc[invalid_geometry_mask]
            .geometry
            .geom_type
            .unique()
            .tolist()
        )
        raise ValueError(
            f"{dataset_name} contains non-point geometries: {invalid_types}"
        )

    if gdf.crs.to_epsg() != 3763:
        logger.info(
            "%s: reprojecting from %s to %s.",
            dataset_name,
            gdf.crs,
            TARGET_CRS,
        )
        gdf = gdf.to_crs(TARGET_CRS)

    if gdf.crs is None or gdf.crs.to_epsg() != 3763:
        raise RuntimeError(
            f"{dataset_name} could not be reprojected to {TARGET_CRS}."
        )

    if required_predictors is None:
        predictors = get_predictor_columns(gdf)
    else:
        gdf = align_required_predictors(
            gdf,
            required_predictors,
            dataset_name,
        )
        predictors = list(required_predictors)

    required_columns = [TARGET_COL, *predictors]

    gdf = convert_required_columns_to_numeric(
        gdf,
        required_columns,
        dataset_name,
    )

    logger.info(
        "%s: %d complete points, %d predictors, CRS=%s.",
        dataset_name,
        len(gdf),
        len(predictors),
        gdf.crs,
    )

    return gdf.reset_index(drop=True), predictors


# ============================================================
# SPATIAL AND MODEL HELPERS
# ============================================================

def spatial_block_arrays(gdf, block_size_m):
    """Calculate block indices and group labels for one block size."""
    if block_size_m <= 0:
        raise ValueError("block_size_m must be greater than zero.")

    block_x = np.floor(
        (gdf.geometry.x.to_numpy() - GRID_OFFSET_X_M)
        / block_size_m
    ).astype(np.int64)

    block_y = np.floor(
        (gdf.geometry.y.to_numpy() - GRID_OFFSET_Y_M)
        / block_size_m
    ).astype(np.int64)

    groups = np.char.add(
        np.char.add(block_x.astype(str), "_"),
        block_y.astype(str),
    )

    return block_x, block_y, groups


def safe_group_kfold(groups, requested_splits, context):
    """Create GroupKFold without exceeding the available groups."""
    number_of_groups = pd.Series(groups).nunique()
    actual_splits = min(int(requested_splits), int(number_of_groups))

    if actual_splits < 2:
        raise ValueError(
            f"{context}: at least two unique spatial blocks are required; "
            f"only {number_of_groups} were found."
        )

    if actual_splits < requested_splits:
        logger.warning(
            "%s: requested %d folds but only %d spatial blocks exist; "
            "using %d folds.",
            context,
            requested_splits,
            number_of_groups,
            actual_splits,
        )

    return GroupKFold(n_splits=actual_splits), actual_splits


def make_rf(config, n_jobs=1):
    """Create RF using the exact settings of one grid combination."""
    return RandomForestRegressor(
        n_estimators=int(config["n_estimators"]),
        max_depth=config["max_depth"],
        min_samples_leaf=int(config["min_samples_leaf"]),
        max_features=config["max_features"],
        bootstrap=bool(config["bootstrap"]),
        random_state=int(config["random_seed"]),
        n_jobs=n_jobs,
    )


def correlation_filter_from_training(X_train, threshold):
    """Remove constant variables and later variables in correlated pairs."""
    if not 0 < threshold <= 1:
        raise ValueError("corr_threshold must be within (0, 1].")

    columns = X_train.columns.tolist()

    constant_features = [
        column
        for column in columns
        if X_train[column].nunique(dropna=True) <= 1
    ]

    candidate_features = [
        column
        for column in columns
        if column not in constant_features
    ]

    removed_features = set()
    records = []

    for column in constant_features:
        records.append({
            "action": "removed_constant",
            "feature": column,
            "paired_with": None,
            "absolute_correlation": np.nan,
        })

    if len(candidate_features) > 1:
        correlation_matrix = (
            X_train[candidate_features]
            .corr(method="pearson")
            .abs()
        )

        for index, retained_feature in enumerate(candidate_features):
            if retained_feature in removed_features:
                continue

            for candidate_for_removal in candidate_features[index + 1:]:
                if candidate_for_removal in removed_features:
                    continue

                correlation_value = correlation_matrix.loc[
                    retained_feature,
                    candidate_for_removal,
                ]

                if (
                    pd.notna(correlation_value)
                    and correlation_value >= threshold
                ):
                    removed_features.add(candidate_for_removal)
                    records.append({
                        "action": "removed_high_correlation",
                        "feature": candidate_for_removal,
                        "paired_with": retained_feature,
                        "absolute_correlation": float(correlation_value),
                    })

    retained_features = [
        column
        for column in candidate_features
        if column not in removed_features
    ]

    if not retained_features:
        raise RuntimeError(
            "Correlation filtering removed all predictor variables."
        )

    report = pd.DataFrame(
        records,
        columns=[
            "action",
            "feature",
            "paired_with",
            "absolute_correlation",
        ],
    )

    return retained_features, report


def fit_rfecv(X_train, y_train, groups_train, config):
    """Run spatial RFECV using one complete parameter combination."""
    inner_cv, actual_folds = safe_group_kfold(
        groups_train,
        int(config["n_spatial_folds"]),
        "Spatial RFECV",
    )

    selector = RFECV(
        estimator=make_rf(config, n_jobs=1),
        step=int(config["rfecv_step"]),
        min_features_to_select=int(config["min_features_to_select"]),
        cv=inner_cv,
        scoring=RFECV_SCORING,
        n_jobs=-1,
        importance_getter=RF_IMPORTANCE_GETTER,
    )

    selector.fit(
        X_train,
        y_train,
        groups=np.asarray(groups_train),
    )

    selected_features = (
        X_train.columns[selector.support_]
        .tolist()
    )

    if not selected_features:
        raise RuntimeError("RFECV did not select any predictor variables.")

    return selector, selected_features, actual_folds


def rfecv_curve_dataframe(selector, config):
    """Convert RFECV cv_results_ into a clean curve table."""
    cv_results = selector.cv_results_

    if "n_features" in cv_results:
        number_of_features = np.asarray(
            cv_results["n_features"],
            dtype=int,
        )
    else:
        # Compatibility fallback for older sklearn versions.
        step = int(config["rfecv_step"])
        minimum = int(config["min_features_to_select"])
        result_length = len(cv_results["mean_test_score"])
        number_of_features = np.arange(
            minimum,
            minimum + result_length * step,
            step,
            dtype=int,
        )[:result_length]

    curve_df = pd.DataFrame({
        "n_features": number_of_features,
        "mean_spatial_CV_RMSE": -np.asarray(
            cv_results["mean_test_score"],
            dtype=np.float64,
        ),
        "std_spatial_CV_score": np.asarray(
            cv_results["std_test_score"],
            dtype=np.float64,
        ),
    })

    return (
        curve_df
        .sort_values("n_features")
        .reset_index(drop=True)
    )


def selected_rfecv_rmse(curve_df, selected_count):
    """Return RFECV mean/std score for the selected feature count."""
    matching = curve_df.loc[
        curve_df["n_features"] == int(selected_count)
    ]

    if matching.empty:
        nearest_index = (
            curve_df["n_features"] - int(selected_count)
        ).abs().idxmin()
        matching = curve_df.loc[[nearest_index]]

    return (
        float(matching["mean_spatial_CV_RMSE"].iloc[0]),
        float(matching["std_spatial_CV_score"].iloc[0]),
    )


# ============================================================
# METRIC HELPERS
# ============================================================

def regression_metrics(y_true, y_pred):
    """Calculate RMSE, MAE, R2, and RPIQ."""
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))

    if len(y_true) >= 2 and np.nanvar(y_true) > 0:
        r2 = float(r2_score(y_true, y_pred))
    else:
        r2 = np.nan

    q25, q75 = np.percentile(y_true, [25, 75])
    iqr = float(q75 - q25)
    rpiq = float(iqr / rmse) if rmse > 0 else np.nan

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "RPIQ": rpiq,
        "Observed_Q25": float(q25),
        "Observed_Q75": float(q75),
        "Observed_IQR": iqr,
    }


def feature_selection_impact_row(
    evaluation,
    n_observations,
    n_all_features,
    n_selected_features,
    all_metrics,
    selected_metrics,
):
    """Create one direct all-features versus selected-features row."""
    feature_reduction = int(n_all_features - n_selected_features)
    feature_reduction_percent = (
        100.0 * feature_reduction / n_all_features
        if n_all_features > 0
        else np.nan
    )

    delta_r2 = selected_metrics["R2"] - all_metrics["R2"]
    rmse_reduction = all_metrics["RMSE"] - selected_metrics["RMSE"]
    mae_reduction = all_metrics["MAE"] - selected_metrics["MAE"]
    delta_rpiq = selected_metrics["RPIQ"] - all_metrics["RPIQ"]

    improved_metrics = sum([
        bool(pd.notna(delta_r2) and delta_r2 > 0),
        bool(pd.notna(rmse_reduction) and rmse_reduction > 0),
        bool(pd.notna(mae_reduction) and mae_reduction > 0),
        bool(pd.notna(delta_rpiq) and delta_rpiq > 0),
    ])

    if improved_metrics == 4:
        interpretation = (
            "Selected features improved R2 and RPIQ and reduced RMSE "
            "and MAE under the same best parameters."
        )
    elif improved_metrics == 0:
        interpretation = (
            "All features performed as well as or better than the selected "
            "features for all reported metrics."
        )
    else:
        interpretation = (
            "The comparison produced mixed results across the reported "
            "metrics; interpret each metric separately."
        )

    return {
        "evaluation": evaluation,
        "n_observations": int(n_observations),
        "n_all_features": int(n_all_features),
        "n_selected_features": int(n_selected_features),
        "features_removed": feature_reduction,
        "feature_reduction_percent": feature_reduction_percent,
        "R2_all_features": all_metrics["R2"],
        "R2_selected_features": selected_metrics["R2"],
        "Delta_R2_selected_minus_all": delta_r2,
        "RMSE_all_features": all_metrics["RMSE"],
        "RMSE_selected_features": selected_metrics["RMSE"],
        "RMSE_reduction_all_minus_selected": rmse_reduction,
        "RMSE_reduction_percent": (
            100.0 * rmse_reduction / all_metrics["RMSE"]
            if all_metrics["RMSE"] > 0
            else np.nan
        ),
        "MAE_all_features": all_metrics["MAE"],
        "MAE_selected_features": selected_metrics["MAE"],
        "MAE_reduction_all_minus_selected": mae_reduction,
        "MAE_reduction_percent": (
            100.0 * mae_reduction / all_metrics["MAE"]
            if all_metrics["MAE"] > 0
            else np.nan
        ),
        "RPIQ_all_features": all_metrics["RPIQ"],
        "RPIQ_selected_features": selected_metrics["RPIQ"],
        "Delta_RPIQ_selected_minus_all": delta_rpiq,
        "interpretation": interpretation,
    }


def save_all_vs_selected_comparison_chart(impact_df, output_png):
    """Save a compact chart comparing all and selected features."""
    metric_specs = [
        ("R2", "R2_all_features", "R2_selected_features"),
        ("RMSE", "RMSE_all_features", "RMSE_selected_features"),
        ("MAE", "MAE_all_features", "MAE_selected_features"),
        ("RPIQ", "RPIQ_all_features", "RPIQ_selected_features"),
    ]

    figure, axes = plt.subplots(2, 2, figsize=(12, 8))
    x = np.arange(len(impact_df))
    width = 0.36
    labels = (
        impact_df["evaluation"]
        .str.replace("_", " ", regex=False)
        .tolist()
    )

    for axis, (metric, all_column, selected_column) in zip(
        axes.flat,
        metric_specs,
    ):
        axis.bar(
            x - width / 2,
            impact_df[all_column],
            width,
            label="All features",
        )
        axis.bar(
            x + width / 2,
            impact_df[selected_column],
            width,
            label="RFECV-selected features",
        )
        axis.set_title(metric)
        axis.set_xticks(x)
        axis.set_xticklabels(labels, rotation=12, ha="right")
        axis.grid(axis="y", alpha=0.25)

    axes.flat[0].legend()
    figure.suptitle(
        "All features versus RFECV-selected features using the same "
        "best parameters",
        fontsize=13,
    )
    figure.tight_layout()
    figure.savefig(
        output_png,
        dpi=OUTPUT_DPI,
        bbox_inches="tight",
    )
    plt.close(figure)


def evaluate_selected_features_spatial_cv(
    X,
    y,
    groups,
    selected_features,
    config,
):
    """
    Evaluate one RFECV-selected subset using out-of-fold spatial predictions.

    The same RF parameters used by RFECV are used in every evaluation fold.
    """
    cv, actual_folds = safe_group_kfold(
        groups,
        int(config["n_spatial_folds"]),
        "Selected-feature spatial CV evaluation",
    )

    oof_predictions = np.full(len(y), np.nan, dtype=np.float64)
    fold_records = []

    for fold_number, (train_indices, test_indices) in enumerate(
        cv.split(X, y, groups),
        start=1,
    ):
        fold_model = make_rf(config, n_jobs=-1)

        fold_model.fit(
            X.iloc[train_indices][selected_features],
            y[train_indices],
        )

        fold_predictions = fold_model.predict(
            X.iloc[test_indices][selected_features]
        )

        oof_predictions[test_indices] = fold_predictions

        fold_metrics = regression_metrics(
            y[test_indices],
            fold_predictions,
        )

        train_groups = set(groups[train_indices].tolist())
        test_groups = set(groups[test_indices].tolist())
        group_overlap = train_groups.intersection(test_groups)

        if group_overlap:
            raise RuntimeError(
                f"Fold {fold_number} contains overlapping spatial blocks."
            )

        fold_records.append({
            "fold": fold_number,
            "n_train": len(train_indices),
            "n_test": len(test_indices),
            "n_train_blocks": len(train_groups),
            "n_test_blocks": len(test_groups),
            "RMSE": fold_metrics["RMSE"],
            "MAE": fold_metrics["MAE"],
            "R2": fold_metrics["R2"],
            "RPIQ": fold_metrics["RPIQ"],
        })

        del fold_model
        gc.collect()

    if np.isnan(oof_predictions).any():
        raise RuntimeError(
            "Some development observations did not receive an "
            "out-of-fold prediction."
        )

    overall_metrics = regression_metrics(y, oof_predictions)
    fold_df = pd.DataFrame(fold_records)

    fold_summary = {
        "mean_fold_RMSE": float(fold_df["RMSE"].mean()),
        "std_fold_RMSE": float(fold_df["RMSE"].std(ddof=1)),
        "mean_fold_MAE": float(fold_df["MAE"].mean()),
        "std_fold_MAE": float(fold_df["MAE"].std(ddof=1)),
        "mean_fold_R2": float(fold_df["R2"].mean()),
        "std_fold_R2": float(fold_df["R2"].std(ddof=1)),
        "mean_fold_RPIQ": float(fold_df["RPIQ"].mean()),
        "std_fold_RPIQ": float(fold_df["RPIQ"].std(ddof=1)),
    }

    return (
        fold_df,
        overall_metrics,
        fold_summary,
        oof_predictions,
        actual_folds,
    )


# ============================================================
# GRID CONSTRUCTION AND VALIDATION
# ============================================================

def build_grid_combinations():
    """Build the Cartesian parameter grid and add stable IDs."""
    required_parameters = set(parameter_columns())
    supplied_parameters = set(PARAMETER_GRID)

    missing = required_parameters.difference(supplied_parameters)
    extra = supplied_parameters.difference(required_parameters)

    if missing:
        raise KeyError(
            "PARAMETER_GRID is missing required parameters: "
            f"{sorted(missing)}"
        )

    if extra:
        raise KeyError(
            "PARAMETER_GRID contains unsupported parameters: "
            f"{sorted(extra)}"
        )

    combinations = []

    for combination_number, config in enumerate(
        ParameterGrid(PARAMETER_GRID),
        start=1,
    ):
        config = dict(config)
        config["parameter_signature"] = combination_signature(config)
        config["grid_id"] = f"G{combination_number:04d}"
        combinations.append(config)

    if not combinations:
        raise ValueError("PARAMETER_GRID produced no combinations.")

    if (
        len(combinations) > MAX_ALLOWED_GRID_COMBINATIONS
        and not ALLOW_LARGE_GRID
    ):
        raise ValueError(
            f"The grid contains {len(combinations):,} combinations, which "
            f"exceeds MAX_ALLOWED_GRID_COMBINATIONS="
            f"{MAX_ALLOWED_GRID_COMBINATIONS}. Reduce PARAMETER_GRID or set "
            "ALLOW_LARGE_GRID=True after confirming the runtime cost."
        )

    return combinations


def grid_plan_dataframe(combinations):
    """Create the parameter table printed before the search starts."""
    rows = []

    for config in combinations:
        row = {
            "grid_id": config["grid_id"],
        }

        for column in parameter_columns():
            row[column] = printable_value(config[column])

        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# SINGLE GRID-COMBINATION EXECUTION
# ============================================================

def run_grid_combination(
    config,
    development_gdf,
    X_raw,
    y_development,
):
    """Run correlation filtering, RFECV, and spatial CV for one row."""
    start_time = time.time()

    _, _, groups = spatial_block_arrays(
        development_gdf,
        int(config["block_size_m"]),
    )

    number_of_blocks = int(pd.Series(groups).nunique())

    correlation_features, _ = correlation_filter_from_training(
        X_raw,
        float(config["corr_threshold"]),
    )

    X_after_correlation = X_raw[correlation_features].copy()

    selector, selected_features, actual_rfecv_folds = fit_rfecv(
        X_after_correlation,
        y_development,
        groups,
        config,
    )

    curve_df = rfecv_curve_dataframe(selector, config)
    rfecv_rmse, rfecv_rmse_std = selected_rfecv_rmse(
        curve_df,
        len(selected_features),
    )

    (
        _,
        cv_metrics,
        fold_summary,
        _,
        actual_evaluation_folds,
    ) = evaluate_selected_features_spatial_cv(
        X_after_correlation,
        y_development,
        groups,
        selected_features,
        config,
    )

    runtime_minutes = (time.time() - start_time) / 60

    result = {
        "grid_id": config["grid_id"],
        "parameter_signature": config["parameter_signature"],
        "status": "success",
        "error_message": "",
        "checkpoint_time_utc": datetime.utcnow().isoformat(
            timespec="seconds"
        ),
        **{
            column: printable_value(config[column])
            for column in parameter_columns()
        },
        "requested_spatial_folds": int(config["n_spatial_folds"]),
        "actual_RFECV_folds": actual_rfecv_folds,
        "actual_evaluation_folds": actual_evaluation_folds,
        "development_spatial_blocks": number_of_blocks,
        "n_raw_features": len(X_raw.columns),
        "n_features_after_correlation": len(correlation_features),
        "n_selected_features": len(selected_features),
        "RFECV_selected_mean_RMSE": rfecv_rmse,
        "RFECV_selected_std_score": rfecv_rmse_std,
        "CV_R2": cv_metrics["R2"],
        "CV_RMSE": cv_metrics["RMSE"],
        "CV_MAE": cv_metrics["MAE"],
        "CV_RPIQ": cv_metrics["RPIQ"],
        **fold_summary,
        "runtime_minutes": runtime_minutes,
        "selected_features": "; ".join(selected_features),
    }

    del selector
    gc.collect()

    return result


# ============================================================
# BEST-COMBINATION DETAILED RUN
# ============================================================

def rerun_and_export_best_combination(
    best_config,
    development_gdf,
    validation_gdf,
    raw_features,
    best_selected_features,
):
    """
    Export the final best-model outputs using the exact feature subset
    stored in the best grid-search result.

    RFECV is rerun only to recreate its diagnostic curve and ranking table.
    The rerun is not allowed to replace the authoritative feature list from
    the best grid-search row.
    """
    logger.info("=" * 90)
    logger.info("RERUNNING BEST PARAMETER COMBINATION FOR FINAL OUTPUTS")
    logger.info("=" * 90)

    y_development = development_gdf[TARGET_COL].to_numpy(dtype=np.float64)
    X_raw = development_gdf[raw_features].copy()

    y_validation = validation_gdf[TARGET_COL].to_numpy(dtype=np.float64)
    X_validation_raw = validation_gdf[raw_features].copy()

    # The selected subset stored in the winning grid-search row is the
    # authoritative subset for final CV, final fitting, validation, SHAP,
    # and the all-versus-selected impact analysis.
    selected_features = list(best_selected_features)

    if not selected_features:
        raise ValueError(
            "The best grid-search row does not contain selected features."
        )

    if len(selected_features) != len(set(selected_features)):
        duplicates = sorted({
            feature
            for feature in selected_features
            if selected_features.count(feature) > 1
        })
        raise ValueError(
            "Duplicate feature names were found in the best grid-search "
            f"feature list: {duplicates}"
        )

    missing_selected_features = [
        feature
        for feature in selected_features
        if feature not in raw_features
    ]

    if missing_selected_features:
        raise ValueError(
            "The best grid-search feature list contains predictors that are "
            "not available in the current development dataset:\n"
            + "\n".join(
                f"  - {feature}"
                for feature in missing_selected_features
            )
            + "\nThe checkpoint may belong to different input data."
        )

    logger.info(
        "Using %d selected features stored in the best grid-search row.",
        len(selected_features),
    )

    block_x, block_y, groups = spatial_block_arrays(
        development_gdf,
        int(best_config["block_size_m"]),
    )

    correlation_features, correlation_report = (
        correlation_filter_from_training(
            X_raw,
            float(best_config["corr_threshold"]),
        )
    )

    X_after_correlation = X_raw[correlation_features].copy()

    selector, rerun_selected_features, actual_rfecv_folds = fit_rfecv(
        X_after_correlation,
        y_development,
        groups,
        best_config,
    )

    curve_df = rfecv_curve_dataframe(selector, best_config)

    if rerun_selected_features != selected_features:
        logger.warning(
            "The diagnostic RFECV rerun selected %d features, whereas the "
            "winning grid-search row stored %d features. The final model, "
            "SHAP analysis, and feature-selection impact comparison will "
            "use the %d features stored in the winning grid-search row.",
            len(rerun_selected_features),
            len(selected_features),
            len(selected_features),
        )
        logger.warning(
            "Features only in stored best subset: %s",
            sorted(set(selected_features) - set(rerun_selected_features)),
        )
        logger.warning(
            "Features only in diagnostic rerun: %s",
            sorted(set(rerun_selected_features) - set(selected_features)),
        )

    (
        fold_metrics_df,
        development_cv_metrics,
        development_fold_summary,
        development_oof_predictions,
        actual_evaluation_folds,
    ) = evaluate_selected_features_spatial_cv(
        X_raw,
        y_development,
        groups,
        selected_features,
        best_config,
    )

    # --------------------------------------------------------
    # Evaluate all original features with the same best RF and
    # spatial-CV settings. No correlation filter or RFECV is used.
    # --------------------------------------------------------
    (
        all_features_fold_metrics_df,
        all_features_development_cv_metrics,
        all_features_development_fold_summary,
        all_features_development_oof_predictions,
        actual_all_features_evaluation_folds,
    ) = evaluate_selected_features_spatial_cv(
        X_raw,
        y_development,
        groups,
        raw_features,
        best_config,
    )

    # The GroupKFold inputs are identical, so both models use the same
    # spatial train/test fold assignments.
    fold_comparison_df = fold_metrics_df.merge(
        all_features_fold_metrics_df,
        on=[
            "fold",
            "n_train",
            "n_test",
            "n_train_blocks",
            "n_test_blocks",
        ],
        suffixes=("_selected_features", "_all_features"),
        validate="one_to_one",
    )

    fold_comparison_df["Delta_R2_selected_minus_all"] = (
        fold_comparison_df["R2_selected_features"]
        - fold_comparison_df["R2_all_features"]
    )
    fold_comparison_df["RMSE_reduction_all_minus_selected"] = (
        fold_comparison_df["RMSE_all_features"]
        - fold_comparison_df["RMSE_selected_features"]
    )
    fold_comparison_df["MAE_reduction_all_minus_selected"] = (
        fold_comparison_df["MAE_all_features"]
        - fold_comparison_df["MAE_selected_features"]
    )
    fold_comparison_df["Delta_RPIQ_selected_minus_all"] = (
        fold_comparison_df["RPIQ_selected_features"]
        - fold_comparison_df["RPIQ_all_features"]
    )

    # --------------------------------------------------------
    # Save correlation and selected-feature tables
    # --------------------------------------------------------
    correlation_report.to_csv(
        OUT_CORRELATION_CSV,
        index=False,
    )

    ranking_map = dict(
        zip(
            correlation_features,
            selector.ranking_.astype(int),
        )
    )

    rerun_support_map = dict(
        zip(
            correlation_features,
            selector.support_.astype(bool),
        )
    )

    stored_best_feature_set = set(selected_features)

    selection_df = pd.DataFrame({
        "feature": raw_features,
        "retained_after_correlation_filter": [
            feature in correlation_features
            for feature in raw_features
        ],
        "selected_by_RFECV": [
            feature in stored_best_feature_set
            for feature in raw_features
        ],
        "selected_by_diagnostic_RFECV_rerun": [
            rerun_support_map.get(feature, False)
            for feature in raw_features
        ],
        "RFECV_rerun_ranking": [
            ranking_map.get(feature, np.nan)
            for feature in raw_features
        ],
    })

    selection_df = (
        selection_df
        .sort_values(
            by=[
                "selected_by_RFECV",
                "selected_by_diagnostic_RFECV_rerun",
                "retained_after_correlation_filter",
                "RFECV_rerun_ranking",
                "feature",
            ],
            ascending=[False, False, False, True, True],
        )
        .reset_index(drop=True)
    )

    selection_df.to_csv(
        OUT_SELECTED_CSV,
        index=False,
    )

    curve_df.to_csv(
        OUT_RFECV_CURVE_CSV,
        index=False,
    )

    fold_metrics_df.to_csv(
        OUT_SPATIAL_FOLD_METRICS_CSV,
        index=False,
    )

    all_features_fold_metrics_df.to_csv(
        OUT_ALL_FEATURES_FOLD_METRICS_CSV,
        index=False,
    )

    fold_comparison_df.to_csv(
        OUT_FOLD_COMPARISON_CSV,
        index=False,
    )

    # --------------------------------------------------------
    # RFECV curve chart
    # --------------------------------------------------------
    plt.figure(figsize=(9, 5.5))

    plt.plot(
        curve_df["n_features"],
        curve_df["mean_spatial_CV_RMSE"],
        marker="o",
        linewidth=1.5,
    )

    lower = (
        curve_df["mean_spatial_CV_RMSE"]
        - curve_df["std_spatial_CV_score"]
    )
    upper = (
        curve_df["mean_spatial_CV_RMSE"]
        + curve_df["std_spatial_CV_score"]
    )

    plt.fill_between(
        curve_df["n_features"],
        lower,
        upper,
        alpha=0.18,
    )

    selected_count = len(selected_features)

    plt.axvline(
        selected_count,
        linestyle="--",
        label=f"Selected features = {selected_count}",
    )

    plt.xlabel("Number of predictor variables")
    plt.ylabel("Mean spatial CV RMSE")
    plt.title("RFECV performance for the best grid-search parameters")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        OUT_RFECV_CURVE_PNG,
        dpi=OUTPUT_DPI,
        bbox_inches="tight",
    )
    plt.close()

    # --------------------------------------------------------
    # Train final model with the exact best grid parameters
    # --------------------------------------------------------
    final_model = make_rf(best_config, n_jobs=-1)

    final_model.fit(
        X_raw[selected_features],
        y_development,
    )

    validation_predictions = final_model.predict(
        X_validation_raw[selected_features]
    )

    validation_metrics = regression_metrics(
        y_validation,
        validation_predictions,
    )

    # All-features final model using exactly the same best Random Forest
    # parameters. Feature-selection parameters are intentionally ignored.
    all_features_model = make_rf(best_config, n_jobs=-1)
    all_features_model.fit(
        X_raw[raw_features],
        y_development,
    )

    all_features_validation_predictions = all_features_model.predict(
        X_validation_raw[raw_features]
    )

    all_features_validation_metrics = regression_metrics(
        y_validation,
        all_features_validation_predictions,
    )

    # --------------------------------------------------------
    # Save metrics summary
    # --------------------------------------------------------
    metrics_summary_df = pd.DataFrame([
        {
            "evaluation": "Development_spatial_cross_validation",
            "model": "All_features_no_feature_selection",
            "n_observations": len(y_development),
            "n_features": len(raw_features),
            "R2": all_features_development_cv_metrics["R2"],
            "RMSE": all_features_development_cv_metrics["RMSE"],
            "MAE": all_features_development_cv_metrics["MAE"],
            "RPIQ": all_features_development_cv_metrics["RPIQ"],
            "mean_fold_R2": all_features_development_fold_summary[
                "mean_fold_R2"
            ],
            "std_fold_R2": all_features_development_fold_summary[
                "std_fold_R2"
            ],
            "mean_fold_RMSE": all_features_development_fold_summary[
                "mean_fold_RMSE"
            ],
            "std_fold_RMSE": all_features_development_fold_summary[
                "std_fold_RMSE"
            ],
        },
        {
            "evaluation": "Development_spatial_cross_validation",
            "model": "Correlation_plus_RFECV_selected_features",
            "n_observations": len(y_development),
            "n_features": len(selected_features),
            "R2": development_cv_metrics["R2"],
            "RMSE": development_cv_metrics["RMSE"],
            "MAE": development_cv_metrics["MAE"],
            "RPIQ": development_cv_metrics["RPIQ"],
            "mean_fold_R2": development_fold_summary["mean_fold_R2"],
            "std_fold_R2": development_fold_summary["std_fold_R2"],
            "mean_fold_RMSE": development_fold_summary["mean_fold_RMSE"],
            "std_fold_RMSE": development_fold_summary["std_fold_RMSE"],
        },
        {
            "evaluation": "Independent_10_percent_validation",
            "model": "All_features_no_feature_selection",
            "n_observations": len(y_validation),
            "n_features": len(raw_features),
            "R2": all_features_validation_metrics["R2"],
            "RMSE": all_features_validation_metrics["RMSE"],
            "MAE": all_features_validation_metrics["MAE"],
            "RPIQ": all_features_validation_metrics["RPIQ"],
            "mean_fold_R2": np.nan,
            "std_fold_R2": np.nan,
            "mean_fold_RMSE": np.nan,
            "std_fold_RMSE": np.nan,
        },
        {
            "evaluation": "Independent_10_percent_validation",
            "model": "Correlation_plus_RFECV_selected_features",
            "n_observations": len(y_validation),
            "n_features": len(selected_features),
            "R2": validation_metrics["R2"],
            "RMSE": validation_metrics["RMSE"],
            "MAE": validation_metrics["MAE"],
            "RPIQ": validation_metrics["RPIQ"],
            "mean_fold_R2": np.nan,
            "std_fold_R2": np.nan,
            "mean_fold_RMSE": np.nan,
            "std_fold_RMSE": np.nan,
        },
    ])

    metrics_summary_df.to_csv(
        OUT_METRICS_SUMMARY_CSV,
        index=False,
    )

    impact_summary_df = pd.DataFrame([
        feature_selection_impact_row(
            evaluation="Development_spatial_cross_validation",
            n_observations=len(y_development),
            n_all_features=len(raw_features),
            n_selected_features=len(selected_features),
            all_metrics=all_features_development_cv_metrics,
            selected_metrics=development_cv_metrics,
        ),
        feature_selection_impact_row(
            evaluation="Independent_10_percent_validation",
            n_observations=len(y_validation),
            n_all_features=len(raw_features),
            n_selected_features=len(selected_features),
            all_metrics=all_features_validation_metrics,
            selected_metrics=validation_metrics,
        ),
    ])

    impact_summary_df.to_csv(
        OUT_FEATURE_SELECTION_IMPACT_CSV,
        index=False,
    )

    comparison_long_df = metrics_summary_df.copy()
    comparison_long_df.to_csv(
        OUT_FEATURE_SELECTION_COMPARISON_CSV,
        index=False,
    )

    save_all_vs_selected_comparison_chart(
        impact_summary_df,
        OUT_FEATURE_SELECTION_COMPARISON_PNG,
    )

    # --------------------------------------------------------
    # Save validation predictions
    # --------------------------------------------------------
    validation_output = validation_gdf.copy()
    validation_output["Pred_RFECV"] = validation_predictions
    validation_output["Residual_RFECV"] = (
        validation_output[TARGET_COL]
        - validation_output["Pred_RFECV"]
    )
    validation_output["Absolute_Error_RFECV"] = np.abs(
        validation_output["Residual_RFECV"]
    )
    validation_output["Pred_AllFeatures"] = (
        all_features_validation_predictions
    )
    validation_output["Residual_AllFeatures"] = (
        validation_output[TARGET_COL]
        - validation_output["Pred_AllFeatures"]
    )
    validation_output["Absolute_Error_AllFeatures"] = np.abs(
        validation_output["Residual_AllFeatures"]
    )

    remove_existing_file(OUT_VALIDATION_PREDICTIONS_GPKG)

    validation_output.to_file(
        OUT_VALIDATION_PREDICTIONS_GPKG,
        layer="independent_validation_predictions",
        driver="GPKG",
        index=False,
    )

    # --------------------------------------------------------
    # Save development fold assignments and OOF predictions
    # --------------------------------------------------------
    development_output = development_gdf.copy()
    development_output["_block_x"] = block_x
    development_output["_block_y"] = block_y
    development_output["_spatial_block"] = groups
    development_output["Pred_CV_RFECV"] = development_oof_predictions
    development_output["Residual_CV_RFECV"] = (
        development_output[TARGET_COL]
        - development_output["Pred_CV_RFECV"]
    )
    development_output["Pred_CV_AllFeatures"] = (
        all_features_development_oof_predictions
    )
    development_output["Residual_CV_AllFeatures"] = (
        development_output[TARGET_COL]
        - development_output["Pred_CV_AllFeatures"]
    )

    map_cv, _ = safe_group_kfold(
        groups,
        int(best_config["n_spatial_folds"]),
        "Final spatial-fold assignment",
    )

    development_output["_spatial_cv_fold"] = -1

    for fold_number, (_, test_indices) in enumerate(
        map_cv.split(X_raw, y_development, groups),
        start=1,
    ):
        development_output.loc[
            test_indices,
            "_spatial_cv_fold",
        ] = fold_number

    remove_existing_file(OUT_BLOCKS_GPKG)

    development_output.to_file(
        OUT_BLOCKS_GPKG,
        layer="spatial_cv_points",
        driver="GPKG",
        index=False,
    )

    # --------------------------------------------------------
    # Random Forest importance
    # --------------------------------------------------------
    rf_importance_df = pd.DataFrame({
        "feature": selected_features,
        "RF_impurity_importance": final_model.feature_importances_,
    })

    rf_importance_df = (
        rf_importance_df
        .sort_values("RF_impurity_importance", ascending=False)
        .reset_index(drop=True)
    )

    rf_importance_df.insert(
        0,
        "rank",
        np.arange(1, len(rf_importance_df) + 1),
    )

    rf_importance_df.to_csv(
        OUT_RF_IMPORTANCE_CSV,
        index=False,
    )

    # --------------------------------------------------------
    # SHAP importance and top-20 beeswarm
    # --------------------------------------------------------
    X_shap = X_raw[selected_features].copy()

    if len(X_shap) > MAX_SHAP_SAMPLES:
        X_shap = X_shap.sample(
            n=MAX_SHAP_SAMPLES,
            random_state=int(best_config["random_seed"]),
        )

    X_shap = X_shap.reset_index(drop=True)

    logger.info(
        "Calculating SHAP values using %d observations and %d features.",
        len(X_shap),
        len(selected_features),
    )

    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer(
        X_shap,
        check_additivity=False,
    )

    shap_array = np.asarray(shap_values.values, dtype=np.float64)

    if shap_array.ndim != 2:
        raise RuntimeError(
            f"Unexpected SHAP value shape: {shap_array.shape}"
        )

    mean_absolute_shap = np.abs(shap_array).mean(axis=0)

    shap_importance_df = pd.DataFrame({
        "feature": selected_features,
        "mean_absolute_SHAP": mean_absolute_shap,
    })

    shap_importance_df = (
        shap_importance_df
        .sort_values("mean_absolute_SHAP", ascending=False)
        .reset_index(drop=True)
    )

    shap_importance_df.insert(
        0,
        "rank",
        np.arange(1, len(shap_importance_df) + 1),
    )

    shap_importance_df.to_csv(
        OUT_SHAP_IMPORTANCE_CSV,
        index=False,
    )

    top_count = min(TOP_N_SHAP_FEATURES, len(shap_importance_df))
    top_shap_df = shap_importance_df.head(top_count).copy()

    top_shap_df.to_csv(
        OUT_SHAP_TOP20_CSV,
        index=False,
    )

    feature_position = {
        feature: position
        for position, feature in enumerate(selected_features)
    }

    top_features = top_shap_df["feature"].tolist()
    top_positions = [feature_position[feature] for feature in top_features]

    top_explanation = shap.Explanation(
        values=shap_array[:, top_positions],
        base_values=shap_values.base_values,
        data=X_shap[top_features].to_numpy(dtype=np.float64),
        feature_names=top_features,
    )

    figure_height = max(7, 0.42 * top_count)

    shap.plots.beeswarm(
        top_explanation,
        max_display=top_count,
        show=False,
        plot_size=(11, figure_height),
    )

    plt.title(
        f"Top {top_count} SHAP predictors for SOC estimation",
        fontsize=14,
        pad=12,
    )
    plt.xlabel("SHAP value (impact on model output)")
    plt.tight_layout()

    plt.savefig(
        OUT_SHAP_PNG,
        dpi=OUTPUT_DPI,
        bbox_inches="tight",
    )
    plt.close()

    # --------------------------------------------------------
    # Save all-features model using the same best RF parameters
    # --------------------------------------------------------
    joblib.dump(
        {
            "model": all_features_model,
            "target_column": TARGET_COL,
            "feature_selection": "none",
            "raw_features": raw_features,
            "best_parameters_from_RFECV_grid_search": {
                key: best_config[key]
                for key in parameter_columns()
            },
            "note": (
                "This all-features model was not tuned separately. It uses "
                "the best parameters selected by the RFECV grid search so "
                "that the effect of feature selection can be compared under "
                "the same model configuration."
            ),
            "development_CV_metrics": (
                all_features_development_cv_metrics
            ),
            "independent_validation_metrics": (
                all_features_validation_metrics
            ),
        },
        OUT_ALL_FEATURES_MODEL,
    )

    # --------------------------------------------------------
    # Save selected-feature final model and metadata
    # --------------------------------------------------------
    joblib.dump(
        {
            "model": final_model,
            "target_column": TARGET_COL,
            "predictor_mode": PREDICTOR_MODE,
            "predictor_prefix": PREDICTOR_PREFIX,
            "excluded_predictor_columns": EXCLUDED_PREDICTOR_COLUMNS,
            "raw_features": raw_features,
            "correlation_retained_features": correlation_features,
            "selected_features": selected_features,
            "selected_features_source": (
                "selected_features stored in the winning grid-search row"
            ),
            "diagnostic_RFECV_rerun_selected_features": (
                rerun_selected_features
            ),
            "best_parameters": {
                key: best_config[key]
                for key in parameter_columns()
            },
            "RFECV_scoring": RFECV_SCORING,
            "target_crs": TARGET_CRS,
            "grid_offset_x_m": GRID_OFFSET_X_M,
            "grid_offset_y_m": GRID_OFFSET_Y_M,
            "development_CV_metrics": development_cv_metrics,
            "independent_validation_metrics": validation_metrics,
            "all_features_development_CV_metrics": (
                all_features_development_cv_metrics
            ),
            "all_features_independent_validation_metrics": (
                all_features_validation_metrics
            ),
        },
        OUT_FINAL_MODEL,
    )

    return {
        "selected_features": selected_features,
        "diagnostic_rerun_selected_features": rerun_selected_features,
        "correlation_features": correlation_features,
        "metrics_summary_df": metrics_summary_df,
        "impact_summary_df": impact_summary_df,
        "shap_importance_df": shap_importance_df,
        "selected_development_cv_metrics": development_cv_metrics,
        "all_features_development_cv_metrics": (
            all_features_development_cv_metrics
        ),
        "selected_validation_metrics": validation_metrics,
        "all_features_validation_metrics": all_features_validation_metrics,
        "actual_rfecv_folds": actual_rfecv_folds,
        "actual_evaluation_folds": actual_evaluation_folds,
        "actual_all_features_evaluation_folds": (
            actual_all_features_evaluation_folds
        ),
    }


# ============================================================
# 1. INITIALIZE AND LOAD DATA
# ============================================================

logger.info("=" * 90)
logger.info("SOC SPATIAL RFECV GRID SEARCH")
logger.info("Run time: %s", datetime.now().isoformat(timespec="seconds"))
logger.info("=" * 90)

all_start_time = time.time()

combinations = build_grid_combinations()
grid_plan_df = grid_plan_dataframe(combinations)

print_dataframe(
    f"PARAMETER GRID: {len(combinations):,} COMBINATIONS",
    grid_plan_df,
    max_rows=100,
)

logger.info(
    "Parameter grid contains %d combinations.",
    len(combinations),
)

development_gdf, raw_features = load_points(
    DEVELOPMENT_GPKG,
    "Development dataset",
)

validation_gdf, validation_features = load_points(
    INDEPENDENT_VALIDATION_GPKG,
    "Independent validation dataset",
    required_predictors=raw_features,
)

if validation_features != raw_features:
    raise RuntimeError(
        "Development and validation predictor order does not match."
    )

X_raw = development_gdf[raw_features].copy()
y_development = development_gdf[TARGET_COL].to_numpy(dtype=np.float64)

# Attribute inventory is saved as part of the final best outputs.
excluded_keys = {
    str(column).casefold()
    for column in EXCLUDED_PREDICTOR_COLUMNS
}

attribute_inventory_df = pd.DataFrame({
    "attribute": development_gdf.columns.astype(str),
    "role": [
        "target"
        if column == TARGET_COL
        else "geometry"
        if column == development_gdf.geometry.name
        else "model_predictor"
        if column in raw_features
        else "excluded_description"
        if str(column).casefold() in excluded_keys
        else "preserved_non_predictor"
        for column in development_gdf.columns
    ],
})

attribute_inventory_df.to_csv(
    OUT_ATTRIBUTE_INVENTORY_CSV,
    index=False,
)


# ============================================================
# 2. RUN OR RESUME THE COMPLETE PARAMETER GRID
# ============================================================

(
    results_records,
    completed_signatures,
    failed_checkpoint_signatures,
) = load_resume_checkpoint(
    OUT_ALL_RESULTS_CSV,
    combinations,
)

current_signature_order = [
    config["parameter_signature"]
    for config in combinations
]

completed_count = len(
    set(current_signature_order).intersection(
        completed_signatures
    )
)

remaining_count = len(combinations) - completed_count

print("\n" + "=" * 120)
print("GRID-SEARCH RESUME STATUS")
print("=" * 120)
print(f"Total current-grid combinations:  {len(combinations):,}")
print(f"Already completed successfully:   {completed_count:,}")
print(f"Combinations remaining:           {remaining_count:,}")
print(
    "Failed/incomplete rows to retry: "
    f"{len(failed_checkpoint_signatures):,}"
)

if remaining_count == 0:
    logger.info(
        "All current-grid combinations are already complete. "
        "Skipping directly to ranking and final best-model export."
    )
else:
    next_config = next(
        config
        for config in combinations
        if config["parameter_signature"] not in completed_signatures
    )

    logger.info(
        "Grid search will resume from %s with parameters: %s",
        next_config["grid_id"],
        {
            key: next_config[key]
            for key in parameter_columns()
        },
    )

# Dictionary-based storage guarantees one latest row per signature.
results_by_signature = {
    record["parameter_signature"]: record
    for record in results_records
    if record.get("parameter_signature")
}

for combination_index, config in enumerate(combinations, start=1):
    signature = config["parameter_signature"]

    if signature in completed_signatures:
        logger.info(
            "Skipping completed %s (%d/%d).",
            config["grid_id"],
            combination_index,
            len(combinations),
        )
        continue

    previous_attempt = results_by_signature.get(signature, {})
    previous_attempt_number = pd.to_numeric(
        pd.Series([
            previous_attempt.get("attempt_number", 0)
        ]),
        errors="coerce",
    ).fillna(0).iloc[0]

    attempt_number = int(previous_attempt_number) + 1

    logger.info("-" * 90)
    logger.info(
        "GRID %d/%d | %s | attempt %d | %s",
        combination_index,
        len(combinations),
        config["grid_id"],
        attempt_number,
        {
            key: config[key]
            for key in parameter_columns()
        },
    )

    try:
        result = run_grid_combination(
            config,
            development_gdf,
            X_raw,
            y_development,
        )

        result["attempt_number"] = attempt_number
        result["grid_position"] = combination_index
        result["grid_total"] = len(combinations)

        logger.info(
            "%s completed | selected=%d | CV R2=%.5f | "
            "CV RMSE=%.5f | runtime=%.2f min",
            config["grid_id"],
            result["n_selected_features"],
            result["CV_R2"],
            result["CV_RMSE"],
            result["runtime_minutes"],
        )

        completed_signatures.add(signature)

    except KeyboardInterrupt:
        logger.warning(
            "%s was interrupted by the user or runtime. The last fully "
            "completed combination remains saved; this combination will "
            "be rerun automatically next time.",
            config["grid_id"],
        )

        # Save all completed rows before allowing the interruption to stop.
        if CHECKPOINT_AFTER_EACH_COMBINATION:
            write_checkpoint_atomic(
                list(results_by_signature.values()),
                OUT_ALL_RESULTS_CSV,
            )

        raise

    except Exception as error:
        logger.exception(
            "%s failed on attempt %d: %s",
            config["grid_id"],
            attempt_number,
            error,
        )

        result = {
            "grid_id": config["grid_id"],
            "parameter_signature": signature,
            "status": "failed",
            "error_message": str(error),
            "checkpoint_time_utc": datetime.utcnow().isoformat(
                timespec="seconds"
            ),
            "attempt_number": attempt_number,
            "grid_position": combination_index,
            "grid_total": len(combinations),
            **{
                column: printable_value(config[column])
                for column in parameter_columns()
            },
            "requested_spatial_folds": int(config["n_spatial_folds"]),
            "actual_RFECV_folds": np.nan,
            "actual_evaluation_folds": np.nan,
            "development_spatial_blocks": np.nan,
            "n_raw_features": len(raw_features),
            "n_features_after_correlation": np.nan,
            "n_selected_features": np.nan,
            "RFECV_selected_mean_RMSE": np.nan,
            "RFECV_selected_std_score": np.nan,
            "CV_R2": np.nan,
            "CV_RMSE": np.nan,
            "CV_MAE": np.nan,
            "CV_RPIQ": np.nan,
            "mean_fold_RMSE": np.nan,
            "std_fold_RMSE": np.nan,
            "mean_fold_MAE": np.nan,
            "std_fold_MAE": np.nan,
            "mean_fold_R2": np.nan,
            "std_fold_R2": np.nan,
            "mean_fold_RPIQ": np.nan,
            "std_fold_RPIQ": np.nan,
            "runtime_minutes": np.nan,
            "selected_features": "",
        }

    # Replace any older row for this combination with the newest attempt.
    results_by_signature[signature] = result
    results_records = list(results_by_signature.values())

    if CHECKPOINT_AFTER_EACH_COMBINATION:
        write_checkpoint_atomic(
            results_records,
            OUT_ALL_RESULTS_CSV,
        )

    completed_now = sum(
        checkpoint_is_successful(record)
        for record in results_records
    )

    logger.info(
        "Checkpoint progress: %d/%d successful combinations.",
        completed_now,
        len(combinations),
    )

    gc.collect()

# Ensure the final checkpoint exists even when checkpoint-per-row is disabled.
results_records = list(results_by_signature.values())

write_checkpoint_atomic(
    results_records,
    OUT_ALL_RESULTS_CSV,
)


# ============================================================
# 3. RANK GRID RESULTS AND SELECT THE BEST COMBINATION
# ============================================================

all_results_df = pd.DataFrame(results_records)

successful_mask = (
    all_results_df["status"].eq("success")
    & pd.to_numeric(all_results_df["CV_R2"], errors="coerce").notna()
    & pd.to_numeric(all_results_df["CV_RMSE"], errors="coerce").notna()
)

successful_df = all_results_df.loc[successful_mask].copy()

if successful_df.empty:
    raise RuntimeError(
        "No parameter combination completed successfully. Review the log "
        f"file:\n{LOG_FILE}"
    )

numeric_ranking_columns = [
    "CV_R2",
    "CV_RMSE",
    "CV_MAE",
    "n_selected_features",
    "runtime_minutes",
]

for column in numeric_ranking_columns:
    successful_df[column] = pd.to_numeric(
        successful_df[column],
        errors="coerce",
    )

successful_df["rank_R2"] = successful_df["CV_R2"].rank(
    method="min",
    ascending=False,
)

successful_df["rank_RMSE"] = successful_df["CV_RMSE"].rank(
    method="min",
    ascending=True,
)

successful_df = (
    successful_df
    .sort_values(
        by=[
            "CV_R2",
            "CV_RMSE",
            "CV_MAE",
            "n_selected_features",
            "runtime_minutes",
        ],
        ascending=[False, True, True, True, True],
    )
    .reset_index(drop=True)
)

successful_df["overall_rank"] = np.arange(
    1,
    len(successful_df) + 1,
)

successful_df["is_best"] = False
successful_df.loc[0, "is_best"] = True

# Add ranks back to the complete results table, including failed rows.
rank_columns = successful_df[
    [
        "parameter_signature",
        "rank_R2",
        "rank_RMSE",
        "overall_rank",
        "is_best",
    ]
]

all_results_df = all_results_df.drop(
    columns=[
        column
        for column in [
            "rank_R2",
            "rank_RMSE",
            "overall_rank",
            "is_best",
        ]
        if column in all_results_df.columns
    ],
    errors="ignore",
)

all_results_df = all_results_df.merge(
    rank_columns,
    on="parameter_signature",
    how="left",
)

all_results_df = all_results_df.sort_values(
    by=["overall_rank", "grid_id"],
    na_position="last",
).reset_index(drop=True)

write_checkpoint_atomic(
    all_results_df.to_dict("records"),
    OUT_ALL_RESULTS_CSV,
)

best_row = successful_df.iloc[0]

best_selected_features = [
    feature.strip()
    for feature in str(best_row["selected_features"]).split(";")
    if feature.strip()
]

expected_best_feature_count = int(
    pd.to_numeric(best_row["n_selected_features"], errors="raise")
)

if len(best_selected_features) != expected_best_feature_count:
    raise RuntimeError(
        "The selected-feature list stored in the best grid-search row does "
        "not match its n_selected_features value. "
        f"Stored count={expected_best_feature_count}, "
        f"parsed names={len(best_selected_features)}."
    )

missing_best_features = [
    feature
    for feature in best_selected_features
    if feature not in raw_features
]

if missing_best_features:
    raise RuntimeError(
        "The winning grid-search row contains features that are absent from "
        "the current input dataset:\n"
        + "\n".join(f"  - {feature}" for feature in missing_best_features)
        + "\nUse the input data associated with this checkpoint or rerun "
        "the grid search in a new output folder."
    )

best_config = {
    "grid_id": best_row["grid_id"],
    "parameter_signature": best_row["parameter_signature"],
}

# Recover exact Python types directly from the original grid object.
config_lookup = {
    config["parameter_signature"]: config
    for config in combinations
}

best_config.update(
    config_lookup[best_row["parameter_signature"]]
)

best_parameters_record = {
    "selection_rule": (
        "Highest development spatial-CV R2; lowest spatial-CV RMSE "
        "as the primary tie-breaker"
    ),
    "grid_id": best_row["grid_id"],
    **{
        column: printable_value(best_config[column])
        for column in parameter_columns()
    },
    "n_features_after_correlation": best_row[
        "n_features_after_correlation"
    ],
    "n_selected_features": len(best_selected_features),
    "CV_R2": best_row["CV_R2"],
    "CV_RMSE": best_row["CV_RMSE"],
    "CV_MAE": best_row["CV_MAE"],
    "CV_RPIQ": best_row["CV_RPIQ"],
    "selected_features": "; ".join(best_selected_features),
}

best_parameters_df = pd.DataFrame([best_parameters_record])
best_parameters_df.to_csv(
    OUT_BEST_PARAMETERS_CSV,
    index=False,
)

print_dataframe(
    "TOP 10 GRID-SEARCH RESULTS",
    successful_df[
        [
            "overall_rank",
            "grid_id",
            *parameter_columns(),
            "n_features_after_correlation",
            "n_selected_features",
            "CV_R2",
            "CV_RMSE",
            "CV_MAE",
            "CV_RPIQ",
            "runtime_minutes",
        ]
    ],
    max_rows=10,
)

print_dataframe(
    "BEST PARAMETER COMBINATION",
    best_parameters_df,
)

logger.info(
    "Best combination: %s | CV R2=%.6f | CV RMSE=%.6f",
    best_row["grid_id"],
    best_row["CV_R2"],
    best_row["CV_RMSE"],
)


# ============================================================
# 4. RERUN THE BEST COMBINATION AND SAVE DETAILED OUTPUTS
# ============================================================

best_artifacts = rerun_and_export_best_combination(
    best_config,
    development_gdf,
    validation_gdf,
    raw_features,
    best_selected_features,
)


# ============================================================
# 5. FINAL REPORT
# ============================================================

print("\n" + "=" * 120)
print("GRID SEARCH AND FINAL BEST-MODEL WORKFLOW COMPLETED")
print("=" * 120)

print(f"Grid combinations evaluated:             {len(combinations):,}")
print(f"Successful combinations:                 {len(successful_df):,}")
print(f"Failed combinations:                     {(~successful_mask).sum():,}")
print(f"Best grid ID:                            {best_row['grid_id']}")
print(f"Best development spatial-CV R2:          {best_row['CV_R2']:.6f}")
print(f"Best development spatial-CV RMSE:        {best_row['CV_RMSE']:.6f}")
print(f"Best development spatial-CV MAE:         {best_row['CV_MAE']:.6f}")
print(f"Best development spatial-CV RPIQ:        {best_row['CV_RPIQ']:.6f}")
print(
    f"Features retained after correlation:     "
    f"{len(best_artifacts['correlation_features']):,}"
)
print(
    f"RFECV-selected features from best row:   "
    f"{len(best_artifacts['selected_features']):,}"
)
print(
    f"Diagnostic RFECV rerun features:         "
    f"{len(best_artifacts['diagnostic_rerun_selected_features']):,}"
)
print(
    f"Total runtime:                           "
    f"{(time.time() - all_start_time) / 3600:.2f} hours"
)

print("\nBest parameters:")
for parameter in parameter_columns():
    print(f"  {parameter}: {best_config[parameter]}")

print("\nBest RFECV-selected feature names:")
for rank, feature in enumerate(
    best_artifacts["selected_features"],
    start=1,
):
    print(f"  {rank:>3}. {feature}")

print_dataframe(
    "ALL FEATURES VS RFECV-SELECTED FEATURES USING THE SAME BEST PARAMETERS",
    best_artifacts["metrics_summary_df"],
)

print_dataframe(
    "IMPACT OF FEATURE SELECTION",
    best_artifacts["impact_summary_df"],
)

print("\n" + "=" * 120)
print("SAVED OUTPUT FILES")
print("=" * 120)

output_files = [
    OUT_ALL_RESULTS_CSV,
    OUT_BEST_PARAMETERS_CSV,
    OUT_ATTRIBUTE_INVENTORY_CSV,
    OUT_CORRELATION_CSV,
    OUT_SELECTED_CSV,
    OUT_RFECV_CURVE_CSV,
    OUT_RFECV_CURVE_PNG,
    OUT_SPATIAL_FOLD_METRICS_CSV,
    OUT_ALL_FEATURES_FOLD_METRICS_CSV,
    OUT_FOLD_COMPARISON_CSV,
    OUT_FEATURE_SELECTION_COMPARISON_CSV,
    OUT_FEATURE_SELECTION_IMPACT_CSV,
    OUT_FEATURE_SELECTION_COMPARISON_PNG,
    OUT_SHAP_IMPORTANCE_CSV,
    OUT_SHAP_TOP20_CSV,
    OUT_RF_IMPORTANCE_CSV,
    OUT_SHAP_PNG,
    OUT_METRICS_SUMMARY_CSV,
    OUT_VALIDATION_PREDICTIONS_GPKG,
    OUT_BLOCKS_GPKG,
    OUT_FINAL_MODEL,
    OUT_ALL_FEATURES_MODEL,
    LOG_FILE,
]

for output_file in output_files:
    print(output_file)


2026-07-28 09:47:06,957 | INFO | 
2026-07-28 09:47:06,959 | INFO | ==========================================================================================
2026-07-28 09:47:06,959 | INFO | NEW RESUME SESSION
2026-07-28 09:47:06,960 | INFO | ==========================================================================================
2026-07-28 09:47:06,970 | INFO | ==========================================================================================
2026-07-28 09:47:06,971 | INFO | SOC SPATIAL RFECV GRID SEARCH
2026-07-28 09:47:06,972 | INFO | Run time: 2026-07-28T09:47:06
2026-07-28 09:47:06,973 | INFO | ==========================================================================================
2026-07-28 09:47:06,980 | INFO | Parameter grid contains 64 combinations.
2026-07-28 09:47:07,035 | INFO | Development dataset: 385 complete points, 64 predictors, CRS=EPSG:3763.
2026-07-28 09:47:07,086 | INFO | Independent validation dataset: 43 complete points, 64 predictors, CRS=EPSG:3763


PARAMETER GRID: 64 COMBINATIONS
grid_id  random_seed  block_size_m  n_spatial_folds  corr_threshold  n_estimators max_depth  min_samples_leaf max_features  bootstrap  rfecv_step  min_features_to_select
  G0001           42         15000                5            0.95          1000      None                 1         sqrt       True           1                       1
  G0002          123         15000                5            0.95          1000      None                 1         sqrt       True           1                       1
  G0003           42         15000               10            0.95          1000      None                 1         sqrt       True           1                       1
  G0004          123         15000               10            0.95          1000      None                 1         sqrt       True           1                       1
  G0005           42         15000                5            0.95          2000      None                 1        

2026-07-28 09:47:07,181 | INFO | Skipping completed G0060 (60/64).
2026-07-28 09:47:07,182 | INFO | Skipping completed G0061 (61/64).
2026-07-28 09:47:07,182 | INFO | Skipping completed G0062 (62/64).
2026-07-28 09:47:07,183 | INFO | Skipping completed G0063 (63/64).
2026-07-28 09:47:07,184 | INFO | Skipping completed G0064 (64/64).
2026-07-28 09:47:07,252 | INFO | Best combination: G0049 | CV R2=0.329828 | CV RMSE=19.358882
2026-07-28 09:47:07,253 | INFO | ==========================================================================================
2026-07-28 09:47:07,254 | INFO | RERUNNING BEST PARAMETER COMBINATION FOR FINAL OUTPUTS
2026-07-28 09:47:07,255 | INFO | ==========================================================================================
2026-07-28 09:47:07,258 | INFO | Using 33 selected features stored in the best grid-search row.



TOP 10 GRID-SEARCH RESULTS
 overall_rank grid_id  random_seed  block_size_m  n_spatial_folds  corr_threshold  n_estimators  max_depth  min_samples_leaf max_features  bootstrap  rfecv_step  min_features_to_select  n_features_after_correlation  n_selected_features    CV_R2   CV_RMSE    CV_MAE  CV_RPIQ  runtime_minutes
            1   G0049           42         20000                5            0.99          1000        NaN                 1         sqrt       True           1                       1                            64                   33 0.329828 19.358882 13.404386 1.270735         4.736678
            2   G0033           42         20000                5            0.95          1000        NaN                 1         sqrt       True           1                       1                            64                   33 0.329828 19.358882 13.404386 1.270735        16.021895
            3   G0050          123         20000                5            0.99          1000    

2026-07-28 09:50:06,350 | WARNING | The diagnostic RFECV rerun selected 53 features, whereas the winning grid-search row stored 33 features. The final model, SHAP analysis, and feature-selection impact comparison will use the 33 features stored in the winning grid-search row.
2026-07-28 09:50:06,351 | WARNING | Features only in stored best subset: []
2026-07-28 09:50:06,352 | WARNING | Features only in diagnostic rerun: ['A03', 'A04', 'A09', 'A11', 'A17', 'A18', 'A20', 'A24', 'A28', 'A36', 'A42', 'A43', 'A46', 'A47', 'A52', 'A55', 'A57', 'A58', 'A60', 'A63']
2026-07-28 09:50:25,331 | INFO | Calculating SHAP values using 385 observations and 33 features.



GRID SEARCH AND FINAL BEST-MODEL WORKFLOW COMPLETED
Grid combinations evaluated:             64
Successful combinations:                 64
Failed combinations:                     0
Best grid ID:                            G0049
Best development spatial-CV R2:          0.329828
Best development spatial-CV RMSE:        19.358882
Best development spatial-CV MAE:         13.404386
Best development spatial-CV RPIQ:        1.270735
Features retained after correlation:     64
RFECV-selected features from best row:   33
Diagnostic RFECV rerun features:         53
Total runtime:                           0.07 hours

Best parameters:
  random_seed: 42
  block_size_m: 20000
  n_spatial_folds: 5
  corr_threshold: 0.99
  n_estimators: 1000
  max_depth: None
  min_samples_leaf: 1
  max_features: sqrt
  bootstrap: True
  rfecv_step: 1
  min_features_to_select: 1

Best RFECV-selected feature names:
    1. A01
    2. A05
    3. A06
    4. A07
    5. A08
    6. A10
    7. A13
    8. A14
    9. A16
  

# RFECV for Sentinel dataset (Dataset-I)

In [3]:
# Optional Colab installation (run once if required):
# !pip install -q geopandas pyogrio shap joblib

"""
Sentinel spatial RFECV and Random Forest workflow for SOC prediction
using one fixed best-parameter configuration.

Comparison design
-----------------
1. The fixed parameters are used to apply the 0.99 correlation filter and
   spatial RFECV once to the complete 90% development dataset.
2. The resulting fixed RFECV-selected feature subset is used for all
   selected-feature model evaluation, final fitting, SHAP analysis, and
   independent validation.
3. A second Random Forest model uses every eligible original predictor with
   no correlation filtering and no RFECV.
4. Both models use exactly the same Random Forest parameters, 20 km spatial
   blocks, five spatial folds, random seed, development observations, and
   independent validation observations.
5. The comparison reports R2, RMSE, MAE, and RPIQ for development spatial
   cross-validation and the independent 10% validation dataset.

Important interpretation
------------------------
The development comparison uses a feature subset selected from the complete
90% development dataset and then evaluates that fixed subset with spatial
cross-validation. The independent 10% validation comparison remains fully
held out and is the strongest assessment of whether feature selection
improves generalization.
"""

import gc
import logging
import time
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFECV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold


# ============================================================
# USER SETTINGS
# ============================================================

DEVELOPMENT_GPKG = (
    "/content/drive/MyDrive/1.SOC_Estimation/"
    "1.Final_SOC_estimation/1_Feature_Selection/Split_features/"
    "LUCAS_samples_sentinel_train_test.gpkg"
)

INDEPENDENT_VALIDATION_GPKG = (
    "/content/drive/MyDrive/1.SOC_Estimation/"
    "1.Final_SOC_estimation/1_Feature_Selection/Split_features/"
    "LUCAS_samples_sentinel_validation.gpkg"
)

OUT_DIR = (
    "/content/drive/MyDrive/1.SOC_Estimation/"
    "1.Final_SOC_estimation/1_Feature_Selection/"
    "RFECV_SOC_Sentinel_Fixed_Best_vs_All_Features"
)

TARGET_COL = "OC"
TARGET_CRS = "EPSG:3763"

# Add any additional non-predictor identifiers (for example POINTID or id)
# to this list if they exist in the GeoPackages. Numeric identifiers must not
# be treated as model predictors.
EXCLUDED_PREDICTOR_COLUMNS = [
    "LC0_Desc",
    "LC1_Desc",
    "LU1_Desc",
]

DROP_ROWS_WITH_NAN = True
FAIL_ON_NONNUMERIC_PREDICTORS = True

GRID_OFFSET_X_M = 0
GRID_OFFSET_Y_M = 0

RFECV_SCORING = "neg_root_mean_squared_error"
RF_IMPORTANCE_GETTER = "auto"

MAX_SHAP_SAMPLES = 1000
TOP_N_SHAP_FEATURES = 20
OUTPUT_DPI = 300


# ============================================================
# FIXED PARAMETERS
# ============================================================

FIXED_PARAMETERS = {
    "random_seed": 42,
    "block_size_m": 20_000,
    "n_spatial_folds": 5,
    "corr_threshold": 0.99,
    "n_estimators": 1000,
    "max_depth": None,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "bootstrap": True,
    "rfecv_step": 1,
    "min_features_to_select": 1,
}


# ============================================================
# OUTPUT FILES
# ============================================================

OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = OUT_DIR / "Sentinel_RFECV_fixed_best_vs_all_features.log"

OUT_FIXED_PARAMETERS_CSV = OUT_DIR / "Sentinel_fixed_parameters.csv"
OUT_ATTRIBUTE_INVENTORY_CSV = (
    OUT_DIR / "Sentinel_fixed_input_attribute_inventory.csv"
)

# Final feature-selection documentation.
OUT_CORRELATION_CSV = (
    OUT_DIR / "Sentinel_final_correlation_filter_results.csv"
)
OUT_SELECTED_CSV = (
    OUT_DIR / "Sentinel_final_RFECV_selected_features.csv"
)
OUT_RFECV_CURVE_CSV = OUT_DIR / "Sentinel_final_RFECV_curve.csv"
OUT_RFECV_CURVE_PNG = OUT_DIR / "Sentinel_final_RFECV_curve.png"

# Feature-selection impact analysis.
OUT_METRICS_SUMMARY_CSV = (
    OUT_DIR / "Sentinel_feature_selection_model_metrics.csv"
)
OUT_IMPACT_SUMMARY_CSV = (
    OUT_DIR / "Sentinel_feature_selection_impact_summary.csv"
)
OUT_SPATIAL_FOLD_METRICS_CSV = (
    OUT_DIR / "Sentinel_feature_selection_spatial_fold_comparison.csv"
)
OUT_SELECTION_FREQUENCY_CSV = (
    OUT_DIR / "Sentinel_RFECV_outer_fold_selection_frequency.csv"
)
OUT_R2_COMPARISON_PNG = OUT_DIR / "Sentinel_feature_selection_R2_comparison.png"
OUT_RMSE_COMPARISON_PNG = OUT_DIR / "Sentinel_feature_selection_RMSE_comparison.png"
OUT_MAE_COMPARISON_PNG = OUT_DIR / "Sentinel_feature_selection_MAE_comparison.png"
OUT_RPIQ_COMPARISON_PNG = OUT_DIR / "Sentinel_feature_selection_RPIQ_comparison.png"

# Feature importance for the final selected-feature model.
OUT_SHAP_IMPORTANCE_CSV = (
    OUT_DIR / "Sentinel_selected_SHAP_feature_importance.csv"
)
OUT_SHAP_TOP20_CSV = (
    OUT_DIR / "Sentinel_selected_SHAP_top20_feature_importance.csv"
)
OUT_RF_IMPORTANCE_CSV = (
    OUT_DIR / "Sentinel_selected_RF_feature_importance.csv"
)
OUT_SHAP_PNG = OUT_DIR / "Sentinel_selected_SHAP_beeswarm_top20.png"

# Predictions, spatial folds, and fitted models.
OUT_VALIDATION_PREDICTIONS_GPKG = (
    OUT_DIR / "Sentinel_feature_selection_independent_validation_predictions.gpkg"
)
OUT_BLOCKS_GPKG = (
    OUT_DIR / "Sentinel_feature_selection_development_spatial_CV_predictions.gpkg"
)
OUT_FINAL_MODEL = OUT_DIR / "Sentinel_RF_selected_features.joblib"
OUT_ALL_VARIABLES_MODEL = OUT_DIR / "Sentinel_RF_all_variables.joblib"

# ============================================================
# LOGGING
# ============================================================

def setup_logger(log_file):
    """Configure console and file logging."""
    logger = logging.getLogger("sentinel_soc_rfecv_feature_selection_impact")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

    file_handler = logging.FileHandler(
        log_file,
        mode="w",
        encoding="utf-8",
    )
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    return logger


logger = setup_logger(LOG_FILE)


# ============================================================
# GENERAL HELPERS
# ============================================================

def print_dataframe(title, dataframe, max_rows=None):
    """Print a DataFrame as a documentation table."""
    print("\n" + "=" * 120)
    print(title)
    print("=" * 120)

    if dataframe.empty:
        print("No records.")
        return

    display_df = dataframe if max_rows is None else dataframe.head(max_rows)

    with pd.option_context(
        "display.max_columns", None,
        "display.width", 300,
        "display.max_colwidth", 80,
    ):
        print(display_df.to_string(index=False))

    if max_rows is not None and len(dataframe) > max_rows:
        print(f"\nShowing {max_rows:,} of {len(dataframe):,} records.")


def printable_value(value):
    """Convert values such as None into clear table text."""
    return "None" if value is None else value


def remove_existing_file(path):
    """Remove an existing file before rewriting it."""
    path = Path(path)
    if path.exists():
        path.unlink()


def build_case_insensitive_lookup(columns):
    """Map case-insensitive names to the actual field names."""
    lookup = {}

    for column in columns:
        key = str(column).strip().casefold()

        if key in lookup:
            raise ValueError(
                "Duplicate field names were found when compared without "
                f"case sensitivity: '{lookup[key]}' and '{column}'."
            )

        lookup[key] = column

    return lookup



def parameter_columns():
    """Return the ordered parameter columns used in output tables."""
    return [
        "random_seed",
        "block_size_m",
        "n_spatial_folds",
        "corr_threshold",
        "n_estimators",
        "max_depth",
        "min_samples_leaf",
        "max_features",
        "bootstrap",
        "rfecv_step",
        "min_features_to_select",
    ]


# ============================================================
# DATA-LOADING HELPERS
# ============================================================

def standardize_target_name(gdf, dataset_name):
    """Find the target field case-insensitively and standardize its name."""
    lookup = build_case_insensitive_lookup(gdf.columns)
    target_key = TARGET_COL.casefold()

    if target_key not in lookup:
        raise ValueError(
            f"{dataset_name} does not contain target field '{TARGET_COL}'."
        )

    actual_target_name = lookup[target_key]

    if actual_target_name != TARGET_COL:
        gdf = gdf.rename(columns={actual_target_name: TARGET_COL})
        logger.warning(
            "%s: renamed target field '%s' to '%s'.",
            dataset_name,
            actual_target_name,
            TARGET_COL,
        )

    return gdf


def get_predictor_columns(gdf):
    """
    Select every available Sentinel attribute except the target, geometry,
    and fields listed in EXCLUDED_PREDICTOR_COLUMNS.

    The selected fields are converted to numeric later by load_points().
    """
    geometry_column = gdf.geometry.name

    excluded_keys = {
        TARGET_COL.casefold(),
        str(geometry_column).strip().casefold(),
        *[
            str(column).strip().casefold()
            for column in EXCLUDED_PREDICTOR_COLUMNS
        ],
    }

    predictors = [
        column
        for column in gdf.columns
        if str(column).strip().casefold() not in excluded_keys
    ]

    if not predictors:
        raise ValueError(
            "No Sentinel predictor columns remain after excluding the "
            "target, geometry, and configured description fields."
        )

    return predictors


def align_required_predictors(gdf, required_predictors, dataset_name):
    """Align validation field names with development predictor names."""
    lookup = build_case_insensitive_lookup(gdf.columns)
    rename_map = {}
    missing = []

    for predictor in required_predictors:
        key = str(predictor).strip().casefold()

        if key not in lookup:
            missing.append(predictor)
            continue

        actual_name = lookup[key]

        if actual_name != predictor:
            rename_map[actual_name] = predictor

    if missing:
        raise ValueError(
            f"{dataset_name} is missing required predictors:\n"
            + "\n".join(f"  - {column}" for column in missing)
        )

    if rename_map:
        gdf = gdf.rename(columns=rename_map)
        logger.warning(
            "%s: aligned field names to development names: %s",
            dataset_name,
            rename_map,
        )

    return gdf


def convert_required_columns_to_numeric(
    gdf,
    required_columns,
    dataset_name,
):
    """Convert target and predictors to finite numeric values."""
    result = gdf.copy()
    conversion_failures = {}

    for column in required_columns:
        original = result[column]
        converted = pd.to_numeric(original, errors="coerce")

        failed_mask = original.notna() & converted.isna()
        failed_count = int(failed_mask.sum())

        if failed_count > 0:
            examples = (
                original.loc[failed_mask]
                .astype(str)
                .drop_duplicates()
                .head(5)
                .tolist()
            )
            conversion_failures[column] = {
                "count": failed_count,
                "examples": examples,
            }

        result[column] = converted.astype(np.float64)

    if conversion_failures:
        lines = [
            f"  - {column}: {details['count']:,} values; "
            f"examples={details['examples']}"
            for column, details in conversion_failures.items()
        ]

        message = (
            f"{dataset_name} contains values that cannot be converted "
            "to numeric form:\n" + "\n".join(lines)
        )

        if FAIL_ON_NONNUMERIC_PREDICTORS:
            raise ValueError(message)

        logger.warning(message)

    result[required_columns] = result[required_columns].replace(
        [np.inf, -np.inf],
        np.nan,
    )

    rows_before = len(result)

    if DROP_ROWS_WITH_NAN:
        result = result.dropna(subset=required_columns).copy()
        rows_removed = rows_before - len(result)

        if rows_removed > 0:
            logger.warning(
                "%s: removed %d rows with missing target or predictors.",
                dataset_name,
                rows_removed,
            )

    elif result[required_columns].isna().any().any():
        missing_counts = result[required_columns].isna().sum()
        missing_counts = missing_counts[missing_counts > 0]

        raise ValueError(
            f"{dataset_name} contains missing target or predictor values:\n"
            + missing_counts.to_string()
        )

    if result.empty:
        raise ValueError(
            f"No complete observations remain in {dataset_name}."
        )

    return result


def load_points(gpkg_path, dataset_name, required_predictors=None):
    """Read, validate, reproject, and prepare a point GeoPackage."""
    gpkg_path = Path(gpkg_path)

    if not gpkg_path.exists():
        raise FileNotFoundError(
            f"{dataset_name} GeoPackage was not found:\n{gpkg_path}"
        )

    gdf = gpd.read_file(gpkg_path)

    if gdf.empty:
        raise ValueError(f"{dataset_name} contains no features.")

    gdf = standardize_target_name(gdf, dataset_name)

    if gdf.crs is None:
        raise ValueError(
            f"{dataset_name} has no CRS information."
        )

    valid_geometry_mask = (
        gdf.geometry.notna()
        & (~gdf.geometry.is_empty)
    )

    removed_geometry_rows = int((~valid_geometry_mask).sum())

    if removed_geometry_rows > 0:
        logger.warning(
            "%s: removed %d rows with missing or empty geometry.",
            dataset_name,
            removed_geometry_rows,
        )

    gdf = gdf.loc[valid_geometry_mask].copy()

    invalid_geometry_mask = ~gdf.geometry.geom_type.isin(["Point"])

    if invalid_geometry_mask.any():
        invalid_types = sorted(
            gdf.loc[invalid_geometry_mask]
            .geometry
            .geom_type
            .unique()
            .tolist()
        )
        raise ValueError(
            f"{dataset_name} contains non-point geometries: {invalid_types}"
        )

    if gdf.crs.to_epsg() != 3763:
        logger.info(
            "%s: reprojecting from %s to %s.",
            dataset_name,
            gdf.crs,
            TARGET_CRS,
        )
        gdf = gdf.to_crs(TARGET_CRS)

    if gdf.crs is None or gdf.crs.to_epsg() != 3763:
        raise RuntimeError(
            f"{dataset_name} could not be reprojected to {TARGET_CRS}."
        )

    if required_predictors is None:
        predictors = get_predictor_columns(gdf)
    else:
        gdf = align_required_predictors(
            gdf,
            required_predictors,
            dataset_name,
        )
        predictors = list(required_predictors)

    required_columns = [TARGET_COL, *predictors]

    gdf = convert_required_columns_to_numeric(
        gdf,
        required_columns,
        dataset_name,
    )

    logger.info(
        "%s: %d complete points, %d predictors, CRS=%s.",
        dataset_name,
        len(gdf),
        len(predictors),
        gdf.crs,
    )

    return gdf.reset_index(drop=True), predictors


# ============================================================
# SPATIAL AND MODEL HELPERS
# ============================================================

def spatial_block_arrays(gdf, block_size_m):
    """Calculate block indices and group labels for one block size."""
    if block_size_m <= 0:
        raise ValueError("block_size_m must be greater than zero.")

    block_x = np.floor(
        (gdf.geometry.x.to_numpy() - GRID_OFFSET_X_M)
        / block_size_m
    ).astype(np.int64)

    block_y = np.floor(
        (gdf.geometry.y.to_numpy() - GRID_OFFSET_Y_M)
        / block_size_m
    ).astype(np.int64)

    groups = np.char.add(
        np.char.add(block_x.astype(str), "_"),
        block_y.astype(str),
    )

    return block_x, block_y, groups


def safe_group_kfold(groups, requested_splits, context):
    """Create GroupKFold without exceeding the available groups."""
    number_of_groups = pd.Series(groups).nunique()
    actual_splits = min(int(requested_splits), int(number_of_groups))

    if actual_splits < 2:
        raise ValueError(
            f"{context}: at least two unique spatial blocks are required; "
            f"only {number_of_groups} were found."
        )

    if actual_splits < requested_splits:
        logger.warning(
            "%s: requested %d folds but only %d spatial blocks exist; "
            "using %d folds.",
            context,
            requested_splits,
            number_of_groups,
            actual_splits,
        )

    return GroupKFold(n_splits=actual_splits), actual_splits


def make_rf(config, n_jobs=1):
    """Create RF using the fixed model parameters."""
    return RandomForestRegressor(
        n_estimators=int(config["n_estimators"]),
        max_depth=config["max_depth"],
        min_samples_leaf=int(config["min_samples_leaf"]),
        max_features=config["max_features"],
        bootstrap=bool(config["bootstrap"]),
        random_state=int(config["random_seed"]),
        n_jobs=n_jobs,
    )


def correlation_filter_from_training(X_train, threshold):
    """Remove constant variables and later variables in correlated pairs."""
    if not 0 < threshold <= 1:
        raise ValueError("corr_threshold must be within (0, 1].")

    columns = X_train.columns.tolist()

    constant_features = [
        column
        for column in columns
        if X_train[column].nunique(dropna=True) <= 1
    ]

    candidate_features = [
        column
        for column in columns
        if column not in constant_features
    ]

    removed_features = set()
    records = []

    for column in constant_features:
        records.append({
            "action": "removed_constant",
            "feature": column,
            "paired_with": None,
            "absolute_correlation": np.nan,
        })

    if len(candidate_features) > 1:
        correlation_matrix = (
            X_train[candidate_features]
            .corr(method="pearson")
            .abs()
        )

        for index, retained_feature in enumerate(candidate_features):
            if retained_feature in removed_features:
                continue

            for candidate_for_removal in candidate_features[index + 1:]:
                if candidate_for_removal in removed_features:
                    continue

                correlation_value = correlation_matrix.loc[
                    retained_feature,
                    candidate_for_removal,
                ]

                if (
                    pd.notna(correlation_value)
                    and correlation_value >= threshold
                ):
                    removed_features.add(candidate_for_removal)
                    records.append({
                        "action": "removed_high_correlation",
                        "feature": candidate_for_removal,
                        "paired_with": retained_feature,
                        "absolute_correlation": float(correlation_value),
                    })

    retained_features = [
        column
        for column in candidate_features
        if column not in removed_features
    ]

    if not retained_features:
        raise RuntimeError(
            "Correlation filtering removed all predictor variables."
        )

    report = pd.DataFrame(
        records,
        columns=[
            "action",
            "feature",
            "paired_with",
            "absolute_correlation",
        ],
    )

    return retained_features, report


def fit_rfecv(X_train, y_train, groups_train, config):
    """Run spatial RFECV using one complete parameter combination."""
    inner_cv, actual_folds = safe_group_kfold(
        groups_train,
        int(config["n_spatial_folds"]),
        "Spatial RFECV",
    )

    selector = RFECV(
        estimator=make_rf(config, n_jobs=1),
        step=int(config["rfecv_step"]),
        min_features_to_select=int(config["min_features_to_select"]),
        cv=inner_cv,
        scoring=RFECV_SCORING,
        n_jobs=-1,
        importance_getter=RF_IMPORTANCE_GETTER,
    )

    selector.fit(
        X_train,
        y_train,
        groups=np.asarray(groups_train),
    )

    selected_features = (
        X_train.columns[selector.support_]
        .tolist()
    )

    if not selected_features:
        raise RuntimeError("RFECV did not select any predictor variables.")

    return selector, selected_features, actual_folds


def rfecv_curve_dataframe(selector, config):
    """Convert RFECV cv_results_ into a clean curve table."""
    cv_results = selector.cv_results_

    if "n_features" in cv_results:
        number_of_features = np.asarray(
            cv_results["n_features"],
            dtype=int,
        )
    else:
        # Compatibility fallback for older sklearn versions.
        step = int(config["rfecv_step"])
        minimum = int(config["min_features_to_select"])
        result_length = len(cv_results["mean_test_score"])
        number_of_features = np.arange(
            minimum,
            minimum + result_length * step,
            step,
            dtype=int,
        )[:result_length]

    curve_df = pd.DataFrame({
        "n_features": number_of_features,
        "mean_spatial_CV_RMSE": -np.asarray(
            cv_results["mean_test_score"],
            dtype=np.float64,
        ),
        "std_spatial_CV_score": np.asarray(
            cv_results["std_test_score"],
            dtype=np.float64,
        ),
    })

    return (
        curve_df
        .sort_values("n_features")
        .reset_index(drop=True)
    )


def selected_rfecv_rmse(curve_df, selected_count):
    """Return RFECV mean/std score for the selected feature count."""
    matching = curve_df.loc[
        curve_df["n_features"] == int(selected_count)
    ]

    if matching.empty:
        nearest_index = (
            curve_df["n_features"] - int(selected_count)
        ).abs().idxmin()
        matching = curve_df.loc[[nearest_index]]

    return (
        float(matching["mean_spatial_CV_RMSE"].iloc[0]),
        float(matching["std_spatial_CV_score"].iloc[0]),
    )


# ============================================================
# METRIC HELPERS
# ============================================================

def regression_metrics(y_true, y_pred):
    """Calculate RMSE, MAE, R2, and RPIQ."""
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))

    if len(y_true) >= 2 and np.nanvar(y_true) > 0:
        r2 = float(r2_score(y_true, y_pred))
    else:
        r2 = np.nan

    q25, q75 = np.percentile(y_true, [25, 75])
    iqr = float(q75 - q25)
    rpiq = float(iqr / rmse) if rmse > 0 else np.nan

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "RPIQ": rpiq,
        "Observed_Q25": float(q25),
        "Observed_Q75": float(q75),
        "Observed_IQR": iqr,
    }


def evaluate_selected_features_spatial_cv(
    X,
    y,
    groups,
    selected_features,
    config,
):
    """
    Evaluate one RFECV-selected subset using out-of-fold spatial predictions.

    The same RF parameters used by RFECV are used in every evaluation fold.
    """
    cv, actual_folds = safe_group_kfold(
        groups,
        int(config["n_spatial_folds"]),
        "Selected-feature spatial CV evaluation",
    )

    oof_predictions = np.full(len(y), np.nan, dtype=np.float64)
    fold_records = []

    for fold_number, (train_indices, test_indices) in enumerate(
        cv.split(X, y, groups),
        start=1,
    ):
        fold_model = make_rf(config, n_jobs=-1)

        fold_model.fit(
            X.iloc[train_indices][selected_features],
            y[train_indices],
        )

        fold_predictions = fold_model.predict(
            X.iloc[test_indices][selected_features]
        )

        oof_predictions[test_indices] = fold_predictions

        fold_metrics = regression_metrics(
            y[test_indices],
            fold_predictions,
        )

        train_groups = set(groups[train_indices].tolist())
        test_groups = set(groups[test_indices].tolist())
        group_overlap = train_groups.intersection(test_groups)

        if group_overlap:
            raise RuntimeError(
                f"Fold {fold_number} contains overlapping spatial blocks."
            )

        fold_records.append({
            "fold": fold_number,
            "n_train": len(train_indices),
            "n_test": len(test_indices),
            "n_train_blocks": len(train_groups),
            "n_test_blocks": len(test_groups),
            "RMSE": fold_metrics["RMSE"],
            "MAE": fold_metrics["MAE"],
            "R2": fold_metrics["R2"],
            "RPIQ": fold_metrics["RPIQ"],
        })

        del fold_model
        gc.collect()

    if np.isnan(oof_predictions).any():
        raise RuntimeError(
            "Some development observations did not receive an "
            "out-of-fold prediction."
        )

    overall_metrics = regression_metrics(y, oof_predictions)
    fold_df = pd.DataFrame(fold_records)

    fold_summary = {
        "mean_fold_RMSE": float(fold_df["RMSE"].mean()),
        "std_fold_RMSE": float(fold_df["RMSE"].std(ddof=1)),
        "mean_fold_MAE": float(fold_df["MAE"].mean()),
        "std_fold_MAE": float(fold_df["MAE"].std(ddof=1)),
        "mean_fold_R2": float(fold_df["R2"].mean()),
        "std_fold_R2": float(fold_df["R2"].std(ddof=1)),
        "mean_fold_RPIQ": float(fold_df["RPIQ"].mean()),
        "std_fold_RPIQ": float(fold_df["RPIQ"].std(ddof=1)),
    }

    return (
        fold_df,
        overall_metrics,
        fold_summary,
        oof_predictions,
        actual_folds,
    )



# ============================================================
# FIXED-PARAMETER VALIDATION
# ============================================================

def validate_fixed_parameters(config):
    """Validate the fixed parameter dictionary before model execution."""
    required = set(parameter_columns())
    supplied = set(config)

    missing = required.difference(supplied)
    extra = supplied.difference(required)

    if missing:
        raise KeyError(
            f"FIXED_PARAMETERS is missing required parameters: {sorted(missing)}"
        )

    if extra:
        raise KeyError(
            f"FIXED_PARAMETERS contains unsupported parameters: {sorted(extra)}"
        )

    if int(config["random_seed"]) < 0:
        raise ValueError("random_seed must be zero or a positive integer.")

    if int(config["block_size_m"]) <= 0:
        raise ValueError("block_size_m must be greater than zero.")

    if int(config["n_spatial_folds"]) < 2:
        raise ValueError("n_spatial_folds must be at least 2.")

    if not 0 < float(config["corr_threshold"]) <= 1:
        raise ValueError("corr_threshold must be within (0, 1].")

    if int(config["n_estimators"]) < 1:
        raise ValueError("n_estimators must be at least 1.")

    if (
        config["max_depth"] is not None
        and int(config["max_depth"]) < 1
    ):
        raise ValueError("max_depth must be None or at least 1.")

    if int(config["min_samples_leaf"]) < 1:
        raise ValueError("min_samples_leaf must be at least 1.")

    if int(config["rfecv_step"]) < 1:
        raise ValueError("rfecv_step must be at least 1.")

    if int(config["min_features_to_select"]) < 1:
        raise ValueError("min_features_to_select must be at least 1.")


def fixed_parameters_dataframe(config):
    """Create a one-row documentation table for the fixed parameters."""
    return pd.DataFrame([
        {
            parameter: printable_value(config[parameter])
            for parameter in parameter_columns()
        }
    ])

# ============================================================
# FIXED-PARAMETER WORKFLOW
# ============================================================


# ============================================================
# FEATURE-SELECTION IMPACT HELPERS
# ============================================================
# ============================================================
# FIXED-SUBSET IMPACT HELPERS
# ============================================================

def materialize_spatial_splits(X, y, groups, config, context):
    """Create one reusable set of spatial folds for both models."""
    cv, actual_folds = safe_group_kfold(
        groups,
        int(config["n_spatial_folds"]),
        context,
    )

    splits = [
        (np.asarray(train_idx), np.asarray(test_idx))
        for train_idx, test_idx in cv.split(X, y, groups)
    ]

    fold_assignment = np.full(len(y), -1, dtype=np.int64)

    for fold_number, (_, test_idx) in enumerate(splits, start=1):
        fold_assignment[test_idx] = fold_number

    if (fold_assignment < 1).any():
        raise RuntimeError(
            "Some development observations were not assigned to a fold."
        )

    return splits, fold_assignment, actual_folds


def evaluate_feature_set_on_splits(
    X,
    y,
    groups,
    feature_names,
    config,
    splits,
    model_label,
):
    """Evaluate one fixed feature set on precomputed spatial folds."""
    missing_features = [
        feature for feature in feature_names
        if feature not in X.columns
    ]

    if missing_features:
        raise ValueError(
            f"{model_label} is missing required predictors: "
            f"{missing_features}"
        )

    oof_predictions = np.full(len(y), np.nan, dtype=np.float64)
    fold_records = []

    for fold_number, (train_idx, test_idx) in enumerate(splits, start=1):
        train_groups = set(groups[train_idx].tolist())
        test_groups = set(groups[test_idx].tolist())
        overlap = train_groups.intersection(test_groups)

        if overlap:
            raise RuntimeError(
                f"{model_label}, fold {fold_number}: overlapping spatial "
                "blocks were found."
            )

        model = make_rf(config, n_jobs=-1)
        model.fit(
            X.iloc[train_idx][feature_names],
            y[train_idx],
        )

        predictions = model.predict(
            X.iloc[test_idx][feature_names]
        )
        oof_predictions[test_idx] = predictions

        metrics = regression_metrics(y[test_idx], predictions)

        fold_records.append({
            "fold": fold_number,
            "model": model_label,
            "n_features": len(feature_names),
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_train_blocks": len(train_groups),
            "n_test_blocks": len(test_groups),
            "R2": metrics["R2"],
            "RMSE": metrics["RMSE"],
            "MAE": metrics["MAE"],
            "RPIQ": metrics["RPIQ"],
        })

        del model
        gc.collect()

    if np.isnan(oof_predictions).any():
        raise RuntimeError(
            f"Some observations did not receive {model_label} predictions."
        )

    fold_df = pd.DataFrame(fold_records)
    overall_metrics = regression_metrics(y, oof_predictions)

    fold_summary = {
        "mean_fold_R2": float(fold_df["R2"].mean()),
        "std_fold_R2": float(fold_df["R2"].std(ddof=1)),
        "mean_fold_RMSE": float(fold_df["RMSE"].mean()),
        "std_fold_RMSE": float(fold_df["RMSE"].std(ddof=1)),
        "mean_fold_MAE": float(fold_df["MAE"].mean()),
        "std_fold_MAE": float(fold_df["MAE"].std(ddof=1)),
        "mean_fold_RPIQ": float(fold_df["RPIQ"].mean()),
        "std_fold_RPIQ": float(fold_df["RPIQ"].std(ddof=1)),
    }

    return {
        "fold_df": fold_df,
        "overall_metrics": overall_metrics,
        "fold_summary": fold_summary,
        "oof_predictions": oof_predictions,
    }


def combine_fold_comparisons(all_result, selected_result):
    """Create a fold-by-fold all-versus-selected comparison table."""
    key_columns = [
        "fold",
        "n_train",
        "n_test",
        "n_train_blocks",
        "n_test_blocks",
    ]

    all_df = all_result["fold_df"].drop(
        columns=["model"]
    ).rename(columns={
        "n_features": "n_features_all",
        "R2": "R2_all",
        "RMSE": "RMSE_all",
        "MAE": "MAE_all",
        "RPIQ": "RPIQ_all",
    })

    selected_df = selected_result["fold_df"].drop(
        columns=["model"]
    ).rename(columns={
        "n_features": "n_features_selected",
        "R2": "R2_selected",
        "RMSE": "RMSE_selected",
        "MAE": "MAE_selected",
        "RPIQ": "RPIQ_selected",
    })

    comparison = all_df.merge(
        selected_df,
        on=key_columns,
        how="inner",
        validate="one_to_one",
    )

    comparison["Delta_R2_Selected_minus_All"] = (
        comparison["R2_selected"] - comparison["R2_all"]
    )
    comparison["RMSE_reduction_All_minus_Selected"] = (
        comparison["RMSE_all"] - comparison["RMSE_selected"]
    )
    comparison["MAE_reduction_All_minus_Selected"] = (
        comparison["MAE_all"] - comparison["MAE_selected"]
    )
    comparison["Delta_RPIQ_Selected_minus_All"] = (
        comparison["RPIQ_selected"] - comparison["RPIQ_all"]
    )

    return comparison


def impact_verdict(r2_all, r2_selected, rmse_all, rmse_selected):
    """Return a transparent interpretation of the comparison."""
    if pd.isna(r2_all) or pd.isna(r2_selected):
        return (
            "Selected features reduced RMSE"
            if rmse_selected < rmse_all
            else "All variables produced lower RMSE"
        )

    r2_improved = r2_selected > r2_all
    rmse_improved = rmse_selected < rmse_all

    if r2_improved and rmse_improved:
        return "Selected features improved both R2 and RMSE"
    if (not r2_improved) and (not rmse_improved):
        return "All variables performed better for both R2 and RMSE"
    return "Mixed result: R2 and RMSE do not agree"


def build_impact_summary(metrics_summary_df):
    """Create a direct all-versus-selected comparison table."""
    rows = []

    for evaluation in metrics_summary_df["evaluation"].drop_duplicates():
        subset = metrics_summary_df.loc[
            metrics_summary_df["evaluation"].eq(evaluation)
        ]

        all_row = subset.loc[
            subset["model"].eq("All_variables_no_feature_selection")
        ].iloc[0]
        selected_row = subset.loc[
            subset["model"].eq("Fixed_RFECV_selected_features")
        ].iloc[0]

        feature_reduction = (
            all_row["n_features"] - selected_row["n_features"]
        )
        feature_reduction_percent = (
            100.0 * feature_reduction / all_row["n_features"]
            if all_row["n_features"] > 0
            else np.nan
        )

        rmse_reduction = all_row["RMSE"] - selected_row["RMSE"]
        mae_reduction = all_row["MAE"] - selected_row["MAE"]

        rows.append({
            "evaluation": evaluation,
            "n_observations": int(all_row["n_observations"]),
            "n_features_all": int(all_row["n_features"]),
            "n_features_selected": int(selected_row["n_features"]),
            "feature_reduction": int(feature_reduction),
            "feature_reduction_percent": feature_reduction_percent,
            "R2_all": all_row["R2"],
            "R2_selected": selected_row["R2"],
            "Delta_R2_Selected_minus_All": (
                selected_row["R2"] - all_row["R2"]
            ),
            "RMSE_all": all_row["RMSE"],
            "RMSE_selected": selected_row["RMSE"],
            "RMSE_reduction_All_minus_Selected": rmse_reduction,
            "RMSE_reduction_percent": (
                100.0 * rmse_reduction / all_row["RMSE"]
                if all_row["RMSE"] > 0
                else np.nan
            ),
            "MAE_all": all_row["MAE"],
            "MAE_selected": selected_row["MAE"],
            "MAE_reduction_All_minus_Selected": mae_reduction,
            "MAE_reduction_percent": (
                100.0 * mae_reduction / all_row["MAE"]
                if all_row["MAE"] > 0
                else np.nan
            ),
            "RPIQ_all": all_row["RPIQ"],
            "RPIQ_selected": selected_row["RPIQ"],
            "Delta_RPIQ_Selected_minus_All": (
                selected_row["RPIQ"] - all_row["RPIQ"]
            ),
            "interpretation": impact_verdict(
                all_row["R2"],
                selected_row["R2"],
                all_row["RMSE"],
                selected_row["RMSE"],
            ),
        })

    return pd.DataFrame(rows)


def save_metric_comparison_plot(metrics_summary_df, metric, output_path):
    """Save one comparison chart for one evaluation metric."""
    plot_df = metrics_summary_df.pivot(
        index="evaluation",
        columns="model",
        values=metric,
    )

    ax = plot_df.plot(
        kind="bar",
        figsize=(10, 5.5),
        rot=0,
    )
    ax.set_xlabel("")
    ax.set_ylabel(metric)
    ax.set_title(f"All variables versus selected variables: {metric}")
    ax.legend(title="Model")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path, dpi=OUTPUT_DPI, bbox_inches="tight")
    plt.close()


# ============================================================
# FIXED BEST-PARAMETER WORKFLOW WITH DIRECT COMPARISON
# ============================================================

def run_fixed_parameter_workflow(
    config,
    development_gdf,
    validation_gdf,
    raw_features,
):
    """
    Select one final feature subset, then compare it with all variables.

    The all-variable model uses no correlation filtering and no RFECV.
    Both models use the same fixed/best Random Forest and spatial settings.
    """
    logger.info("=" * 90)
    logger.info("RUNNING FIXED BEST-PARAMETER FEATURE COMPARISON")
    logger.info("=" * 90)

    y_development = development_gdf[TARGET_COL].to_numpy(dtype=np.float64)
    X_raw = development_gdf[raw_features].copy()

    y_validation = validation_gdf[TARGET_COL].to_numpy(dtype=np.float64)
    X_validation_raw = validation_gdf[raw_features].copy()

    block_x, block_y, groups = spatial_block_arrays(
        development_gdf,
        int(config["block_size_m"]),
    )

    # --------------------------------------------------------
    # 1. Select one final feature subset using the fixed parameters
    # --------------------------------------------------------
    correlation_features, correlation_report = (
        correlation_filter_from_training(
            X_raw,
            float(config["corr_threshold"]),
        )
    )
    X_after_correlation = X_raw[correlation_features].copy()

    selector, selected_features, actual_rfecv_folds = fit_rfecv(
        X_after_correlation,
        y_development,
        groups,
        config,
    )

    if not selected_features:
        raise RuntimeError("RFECV returned an empty feature subset.")

    logger.info(
        "Fixed RFECV subset: %d of %d original variables.",
        len(selected_features),
        len(raw_features),
    )

    curve_df = rfecv_curve_dataframe(selector, config)
    correlation_report.to_csv(OUT_CORRELATION_CSV, index=False)
    curve_df.to_csv(OUT_RFECV_CURVE_CSV, index=False)

    ranking_map = dict(
        zip(correlation_features, selector.ranking_.astype(int))
    )
    support_map = dict(
        zip(correlation_features, selector.support_.astype(bool))
    )

    selection_df = pd.DataFrame({
        "feature": raw_features,
        "retained_after_correlation_filter": [
            feature in correlation_features
            for feature in raw_features
        ],
        "selected_by_fixed_RFECV": [
            support_map.get(feature, False)
            for feature in raw_features
        ],
        "RFECV_ranking": [
            ranking_map.get(feature, np.nan)
            for feature in raw_features
        ],
    })
    selection_df = selection_df.sort_values(
        by=[
            "selected_by_fixed_RFECV",
            "retained_after_correlation_filter",
            "RFECV_ranking",
            "feature",
        ],
        ascending=[False, False, True, True],
    ).reset_index(drop=True)
    selection_df.to_csv(OUT_SELECTED_CSV, index=False)

    # RFECV curve.
    plt.figure(figsize=(9, 5.5))
    plt.plot(
        curve_df["n_features"],
        curve_df["mean_spatial_CV_RMSE"],
        marker="o",
        linewidth=1.5,
    )
    lower = (
        curve_df["mean_spatial_CV_RMSE"]
        - curve_df["std_spatial_CV_score"]
    )
    upper = (
        curve_df["mean_spatial_CV_RMSE"]
        + curve_df["std_spatial_CV_score"]
    )
    plt.fill_between(
        curve_df["n_features"],
        lower,
        upper,
        alpha=0.18,
    )
    plt.axvline(
        len(selected_features),
        linestyle="--",
        label=f"Selected features = {len(selected_features)}",
    )
    plt.xlabel("Number of predictor variables")
    plt.ylabel("Mean spatial CV RMSE")
    plt.title("RFECV performance using the fixed best parameters")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_RFECV_CURVE_PNG, dpi=OUTPUT_DPI, bbox_inches="tight")
    plt.close()

    # --------------------------------------------------------
    # 2. Compare the two fixed feature sets on identical folds
    # --------------------------------------------------------
    splits, fold_assignment, actual_evaluation_folds = (
        materialize_spatial_splits(
            X_raw,
            y_development,
            groups,
            config,
            "All-versus-selected spatial CV",
        )
    )

    all_cv = evaluate_feature_set_on_splits(
        X=X_raw,
        y=y_development,
        groups=groups,
        feature_names=raw_features,
        config=config,
        splits=splits,
        model_label="All_variables_no_feature_selection",
    )

    selected_cv = evaluate_feature_set_on_splits(
        X=X_raw,
        y=y_development,
        groups=groups,
        feature_names=selected_features,
        config=config,
        splits=splits,
        model_label="Fixed_RFECV_selected_features",
    )

    fold_comparison_df = combine_fold_comparisons(
        all_cv,
        selected_cv,
    )
    fold_comparison_df.to_csv(
        OUT_SPATIAL_FOLD_METRICS_CSV,
        index=False,
    )

    # --------------------------------------------------------
    # 3. Fit both final models and evaluate independent validation
    # --------------------------------------------------------
    all_variables_model = make_rf(config, n_jobs=-1)
    all_variables_model.fit(
        X_raw[raw_features],
        y_development,
    )
    validation_predictions_all = all_variables_model.predict(
        X_validation_raw[raw_features]
    )
    validation_metrics_all = regression_metrics(
        y_validation,
        validation_predictions_all,
    )

    selected_model = make_rf(config, n_jobs=-1)
    selected_model.fit(
        X_raw[selected_features],
        y_development,
    )
    validation_predictions_selected = selected_model.predict(
        X_validation_raw[selected_features]
    )
    validation_metrics_selected = regression_metrics(
        y_validation,
        validation_predictions_selected,
    )

    # --------------------------------------------------------
    # 4. Save metrics and impact summaries
    # --------------------------------------------------------
    metrics_summary_df = pd.DataFrame([
        {
            "evaluation": "Development_spatial_cross_validation_fixed_subset",
            "model": "All_variables_no_feature_selection",
            "n_observations": len(y_development),
            "n_features": len(raw_features),
            **all_cv["overall_metrics"],
            **all_cv["fold_summary"],
        },
        {
            "evaluation": "Development_spatial_cross_validation_fixed_subset",
            "model": "Fixed_RFECV_selected_features",
            "n_observations": len(y_development),
            "n_features": len(selected_features),
            **selected_cv["overall_metrics"],
            **selected_cv["fold_summary"],
        },
        {
            "evaluation": "Independent_10_percent_validation",
            "model": "All_variables_no_feature_selection",
            "n_observations": len(y_validation),
            "n_features": len(raw_features),
            **validation_metrics_all,
            "mean_fold_R2": np.nan,
            "std_fold_R2": np.nan,
            "mean_fold_RMSE": np.nan,
            "std_fold_RMSE": np.nan,
            "mean_fold_MAE": np.nan,
            "std_fold_MAE": np.nan,
            "mean_fold_RPIQ": np.nan,
            "std_fold_RPIQ": np.nan,
        },
        {
            "evaluation": "Independent_10_percent_validation",
            "model": "Fixed_RFECV_selected_features",
            "n_observations": len(y_validation),
            "n_features": len(selected_features),
            **validation_metrics_selected,
            "mean_fold_R2": np.nan,
            "std_fold_R2": np.nan,
            "mean_fold_RMSE": np.nan,
            "std_fold_RMSE": np.nan,
            "mean_fold_MAE": np.nan,
            "std_fold_MAE": np.nan,
            "mean_fold_RPIQ": np.nan,
            "std_fold_RPIQ": np.nan,
        },
    ])
    metrics_summary_df.to_csv(OUT_METRICS_SUMMARY_CSV, index=False)

    impact_summary_df = build_impact_summary(metrics_summary_df)
    impact_summary_df.to_csv(OUT_IMPACT_SUMMARY_CSV, index=False)

    save_metric_comparison_plot(
        metrics_summary_df,
        "R2",
        OUT_R2_COMPARISON_PNG,
    )
    save_metric_comparison_plot(
        metrics_summary_df,
        "RMSE",
        OUT_RMSE_COMPARISON_PNG,
    )
    save_metric_comparison_plot(
        metrics_summary_df,
        "MAE",
        OUT_MAE_COMPARISON_PNG,
    )
    save_metric_comparison_plot(
        metrics_summary_df,
        "RPIQ",
        OUT_RPIQ_COMPARISON_PNG,
    )

    # --------------------------------------------------------
    # 5. Save independent-validation predictions
    # --------------------------------------------------------
    validation_output = validation_gdf.copy()
    validation_output["Pred_All"] = validation_predictions_all
    validation_output["Residual_All"] = (
        validation_output[TARGET_COL] - validation_output["Pred_All"]
    )
    validation_output["Abs_Error_All"] = np.abs(
        validation_output["Residual_All"]
    )
    validation_output["Pred_Selected"] = validation_predictions_selected
    validation_output["Residual_Selected"] = (
        validation_output[TARGET_COL]
        - validation_output["Pred_Selected"]
    )
    validation_output["Abs_Error_Selected"] = np.abs(
        validation_output["Residual_Selected"]
    )
    validation_output["Abs_Error_Improvement"] = (
        validation_output["Abs_Error_All"]
        - validation_output["Abs_Error_Selected"]
    )

    remove_existing_file(OUT_VALIDATION_PREDICTIONS_GPKG)
    validation_output.to_file(
        OUT_VALIDATION_PREDICTIONS_GPKG,
        layer="independent_validation_comparison",
        driver="GPKG",
        index=False,
    )

    # --------------------------------------------------------
    # 6. Save development folds and out-of-fold predictions
    # --------------------------------------------------------
    development_output = development_gdf.copy()
    development_output["_block_x"] = block_x
    development_output["_block_y"] = block_y
    development_output["_spatial_block"] = groups
    development_output["_spatial_cv_fold"] = fold_assignment
    development_output["Pred_CV_All"] = all_cv["oof_predictions"]
    development_output["Residual_CV_All"] = (
        development_output[TARGET_COL]
        - development_output["Pred_CV_All"]
    )
    development_output["Pred_CV_Selected"] = (
        selected_cv["oof_predictions"]
    )
    development_output["Residual_CV_Selected"] = (
        development_output[TARGET_COL]
        - development_output["Pred_CV_Selected"]
    )

    remove_existing_file(OUT_BLOCKS_GPKG)
    development_output.to_file(
        OUT_BLOCKS_GPKG,
        layer="fixed_subset_spatial_cv_comparison",
        driver="GPKG",
        index=False,
    )

    # --------------------------------------------------------
    # 7. Importance and SHAP for the selected-feature model
    # --------------------------------------------------------
    rf_importance_df = pd.DataFrame({
        "feature": selected_features,
        "RF_impurity_importance": selected_model.feature_importances_,
    }).sort_values(
        "RF_impurity_importance",
        ascending=False,
    ).reset_index(drop=True)
    rf_importance_df.insert(
        0,
        "rank",
        np.arange(1, len(rf_importance_df) + 1),
    )
    rf_importance_df.to_csv(OUT_RF_IMPORTANCE_CSV, index=False)

    X_shap = X_raw[selected_features].copy()
    if len(X_shap) > MAX_SHAP_SAMPLES:
        X_shap = X_shap.sample(
            n=MAX_SHAP_SAMPLES,
            random_state=int(config["random_seed"]),
        )
    X_shap = X_shap.reset_index(drop=True)

    logger.info(
        "Calculating SHAP values using %d observations and %d features.",
        len(X_shap),
        len(selected_features),
    )

    explainer = shap.TreeExplainer(selected_model)
    shap_values = explainer(X_shap, check_additivity=False)
    shap_array = np.asarray(shap_values.values, dtype=np.float64)

    if shap_array.ndim != 2:
        raise RuntimeError(
            f"Unexpected SHAP value shape: {shap_array.shape}"
        )

    shap_importance_df = pd.DataFrame({
        "feature": selected_features,
        "mean_absolute_SHAP": np.abs(shap_array).mean(axis=0),
    }).sort_values(
        "mean_absolute_SHAP",
        ascending=False,
    ).reset_index(drop=True)
    shap_importance_df.insert(
        0,
        "rank",
        np.arange(1, len(shap_importance_df) + 1),
    )
    shap_importance_df.to_csv(OUT_SHAP_IMPORTANCE_CSV, index=False)

    top_count = min(TOP_N_SHAP_FEATURES, len(shap_importance_df))
    top_shap_df = shap_importance_df.head(top_count).copy()
    top_shap_df.to_csv(OUT_SHAP_TOP20_CSV, index=False)

    feature_position = {
        feature: position
        for position, feature in enumerate(selected_features)
    }
    top_features = top_shap_df["feature"].tolist()
    top_positions = [feature_position[feature] for feature in top_features]

    top_explanation = shap.Explanation(
        values=shap_array[:, top_positions],
        base_values=shap_values.base_values,
        data=X_shap[top_features].to_numpy(dtype=np.float64),
        feature_names=top_features,
    )

    figure_height = max(7, 0.42 * top_count)
    shap.plots.beeswarm(
        top_explanation,
        max_display=top_count,
        show=False,
        plot_size=(11, figure_height),
    )
    plt.title(
        f"Top {top_count} Sentinel SHAP predictors for SOC estimation",
        fontsize=14,
        pad=12,
    )
    plt.xlabel("SHAP value (impact on model output)")
    plt.tight_layout()
    plt.savefig(OUT_SHAP_PNG, dpi=OUTPUT_DPI, bbox_inches="tight")
    plt.close()

    # --------------------------------------------------------
    # 8. Save both fitted models and metadata
    # --------------------------------------------------------
    shared_metadata = {
        "target_column": TARGET_COL,
        "excluded_predictor_columns": EXCLUDED_PREDICTOR_COLUMNS,
        "fixed_best_parameters": {
            key: config[key]
            for key in parameter_columns()
        },
        "RFECV_scoring": RFECV_SCORING,
        "target_crs": TARGET_CRS,
        "grid_offset_x_m": GRID_OFFSET_X_M,
        "grid_offset_y_m": GRID_OFFSET_Y_M,
    }

    joblib.dump(
        {
            "model": selected_model,
            **shared_metadata,
            "model_type": "Fixed_RFECV_selected_features",
            "selection_scope": "complete_90_percent_development_dataset",
            "raw_features": raw_features,
            "correlation_retained_features": correlation_features,
            "selected_features": selected_features,
            "development_spatial_CV_metrics": (
                selected_cv["overall_metrics"]
            ),
            "independent_validation_metrics": validation_metrics_selected,
        },
        OUT_FINAL_MODEL,
    )

    joblib.dump(
        {
            "model": all_variables_model,
            **shared_metadata,
            "model_type": "All_variables_no_feature_selection",
            "features": raw_features,
            "correlation_filter_applied": False,
            "RFECV_applied": False,
            "development_spatial_CV_metrics": all_cv["overall_metrics"],
            "independent_validation_metrics": validation_metrics_all,
            "note": (
                "The all-variable model uses the same fixed Random Forest "
                "parameters as the selected-feature model. Correlation and "
                "RFECV settings are not applied to this model."
            ),
        },
        OUT_ALL_VARIABLES_MODEL,
    )

    del selector
    gc.collect()

    return {
        "selected_features": selected_features,
        "correlation_features": correlation_features,
        "metrics_summary_df": metrics_summary_df,
        "impact_summary_df": impact_summary_df,
        "fold_comparison_df": fold_comparison_df,
        "shap_importance_df": shap_importance_df,
        "actual_rfecv_folds": actual_rfecv_folds,
        "actual_evaluation_folds": actual_evaluation_folds,
    }


def main():
    # ============================================================
    # 1. INITIALIZE AND LOAD DATA
    # ============================================================
    logger.info("=" * 90)
    logger.info("SOC FIXED BEST-PARAMETER FEATURE COMPARISON")
    logger.info("Run time: %s", datetime.now().isoformat(timespec="seconds"))
    logger.info("=" * 90)

    all_start_time = time.time()

    validate_fixed_parameters(FIXED_PARAMETERS)

    fixed_parameters_df = fixed_parameters_dataframe(FIXED_PARAMETERS)
    fixed_parameters_df.to_csv(OUT_FIXED_PARAMETERS_CSV, index=False)

    print_dataframe(
        "FIXED BEST MODEL AND RFECV PARAMETERS",
        fixed_parameters_df,
    )

    development_gdf, raw_features = load_points(
        DEVELOPMENT_GPKG,
        "Development dataset",
    )

    validation_gdf, validation_features = load_points(
        INDEPENDENT_VALIDATION_GPKG,
        "Independent validation dataset",
        required_predictors=raw_features,
    )

    if validation_features != raw_features:
        raise RuntimeError(
            "Development and validation predictor order does not match."
        )

    excluded_keys = {
        str(column).casefold()
        for column in EXCLUDED_PREDICTOR_COLUMNS
    }

    attribute_inventory_df = pd.DataFrame({
        "attribute": development_gdf.columns.astype(str),
        "role": [
            "target"
            if column == TARGET_COL
            else "geometry"
            if column == development_gdf.geometry.name
            else "model_predictor"
            if column in raw_features
            else "excluded_description"
            if str(column).casefold() in excluded_keys
            else "preserved_non_predictor"
            for column in development_gdf.columns
        ],
    })
    attribute_inventory_df.to_csv(
        OUT_ATTRIBUTE_INVENTORY_CSV,
        index=False,
    )

    # ============================================================
    # 2. RUN FIXED BEST-PARAMETER COMPARISON
    # ============================================================
    artifacts = run_fixed_parameter_workflow(
        FIXED_PARAMETERS,
        development_gdf,
        validation_gdf,
        raw_features,
    )

    # ============================================================
    # 3. FINAL REPORT
    # ============================================================
    metrics_df = artifacts["metrics_summary_df"]
    impact_df = artifacts["impact_summary_df"]

    print("\n" + "=" * 120)
    print("FIXED BEST-PARAMETER FEATURE COMPARISON COMPLETED")
    print("=" * 120)

    print(f"Development observations:                {len(development_gdf):,}")
    print(f"Independent validation observations:     {len(validation_gdf):,}")
    print(f"Initial candidate predictors:            {len(raw_features):,}")
    print(
        f"Features retained after correlation:     "
        f"{len(artifacts['correlation_features']):,}"
    )
    print(
        f"Fixed RFECV-selected features:           "
        f"{len(artifacts['selected_features']):,}"
    )
    print(
        f"RFECV spatial folds:                    "
        f"{artifacts['actual_rfecv_folds']}"
    )
    print(
        f"Comparison spatial folds:               "
        f"{artifacts['actual_evaluation_folds']}"
    )
    print(
        f"Total runtime:                           "
        f"{(time.time() - all_start_time) / 3600:.2f} hours"
    )

    print_dataframe(
        "ALL VARIABLES VS FIXED RFECV-SELECTED FEATURES",
        metrics_df[
            [
                "evaluation",
                "model",
                "n_observations",
                "n_features",
                "R2",
                "RMSE",
                "MAE",
                "RPIQ",
                "mean_fold_R2",
                "std_fold_R2",
                "mean_fold_RMSE",
                "std_fold_RMSE",
            ]
        ],
    )

    print_dataframe(
        "IMPACT OF FEATURE SELECTION",
        impact_df,
    )

    print("\nFixed RFECV-selected feature names:")
    for rank, feature in enumerate(
        artifacts["selected_features"],
        start=1,
    ):
        print(f"  {rank:>3}. {feature}")

    print("\n" + "=" * 120)
    print("SAVED OUTPUT FILES")
    print("=" * 120)

    output_files = [
        OUT_FIXED_PARAMETERS_CSV,
        OUT_ATTRIBUTE_INVENTORY_CSV,
        OUT_CORRELATION_CSV,
        OUT_SELECTED_CSV,
        OUT_RFECV_CURVE_CSV,
        OUT_RFECV_CURVE_PNG,
        OUT_METRICS_SUMMARY_CSV,
        OUT_IMPACT_SUMMARY_CSV,
        OUT_SPATIAL_FOLD_METRICS_CSV,
        OUT_R2_COMPARISON_PNG,
        OUT_RMSE_COMPARISON_PNG,
        OUT_MAE_COMPARISON_PNG,
        OUT_RPIQ_COMPARISON_PNG,
        OUT_SHAP_IMPORTANCE_CSV,
        OUT_SHAP_TOP20_CSV,
        OUT_RF_IMPORTANCE_CSV,
        OUT_SHAP_PNG,
        OUT_VALIDATION_PREDICTIONS_GPKG,
        OUT_BLOCKS_GPKG,
        OUT_FINAL_MODEL,
        OUT_ALL_VARIABLES_MODEL,
        LOG_FILE,
    ]

    for output_file in output_files:
        print(output_file)


if __name__ == "__main__":
    main()


2026-07-28 10:22:43,329 | INFO | ==========================================================================================
2026-07-28 10:22:43,331 | INFO | SOC FIXED BEST-PARAMETER FEATURE COMPARISON
2026-07-28 10:22:43,332 | INFO | Run time: 2026-07-28T10:22:43
2026-07-28 10:22:43,333 | INFO | ==========================================================================================



FIXED BEST MODEL AND RFECV PARAMETERS
 random_seed  block_size_m  n_spatial_folds  corr_threshold  n_estimators max_depth  min_samples_leaf max_features  bootstrap  rfecv_step  min_features_to_select
          42         20000                5            0.99          1000      None                 1         sqrt       True           1                       1


2026-07-28 10:22:46,580 | INFO | Development dataset: 385 complete points, 40 predictors, CRS=EPSG:3763.
2026-07-28 10:22:47,006 | INFO | Independent validation dataset: 43 complete points, 40 predictors, CRS=EPSG:3763.
2026-07-28 10:22:47,016 | INFO | ==========================================================================================
2026-07-28 10:22:47,016 | INFO | RUNNING FIXED BEST-PARAMETER FEATURE COMPARISON
2026-07-28 10:22:47,017 | INFO | ==========================================================================================
2026-07-28 10:25:05,930 | INFO | Fixed RFECV subset: 16 of 40 original variables.
2026-07-28 10:25:25,032 | INFO | Calculating SHAP values using 385 observations and 16 features.



FIXED BEST-PARAMETER FEATURE COMPARISON COMPLETED
Development observations:                385
Independent validation observations:     43
Initial candidate predictors:            40
Features retained after correlation:     39
Fixed RFECV-selected features:           16
RFECV spatial folds:                    5
Comparison spatial folds:               5
Total runtime:                           0.05 hours

ALL VARIABLES VS FIXED RFECV-SELECTED FEATURES
                                       evaluation                              model  n_observations  n_features       R2      RMSE       MAE     RPIQ  mean_fold_R2  std_fold_R2  mean_fold_RMSE  std_fold_RMSE
Development_spatial_cross_validation_fixed_subset All_variables_no_feature_selection             385          40 0.285502 19.988836 13.779923 1.230687      0.281179     0.119265       19.823326       2.869954
Development_spatial_cross_validation_fixed_subset      Fixed_RFECV_selected_features             385          16 0.313772 19.5